# Munnar Drone Geolocation

## AI-Based Drone Image Geolocation Using Satellite Image Matching

This cleaned notebook contains the main reproducible pipeline and the verified Munnar validation/output sections from the original Kaggle notebook.

**Pipeline:** Munnar drone image → 384-D AI feature → satellite tile features → cosine matching → local refinement → ground-truth validation → final visualization and graph.

> Note: The final visualization cells use the already verified project result for the reported Munnar coordinates; they do not rerun the AI model.

## 1. Environment and dependencies

In [ ]:
!pip -q install huggingface_hub pystac-client


## 2. Project structure and Munnar input

In [ ]:
# ============================================================
# GENERIC DRONE GEOLOCATION PIPELINE — PROJECT STRUCTURE
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path("/kaggle/working/drone_geolocation_pipeline")

DIRECTORIES = [
    PROJECT_ROOT / "input",
    PROJECT_ROOT / "satellite",
    PROJECT_ROOT / "tiles",
    PROJECT_ROOT / "features",
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "models",
]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

# Existing Munnar aerial image
MUNNAR_IMAGE = Path(
    "/kaggle/input/datasets/vishalswamy2004/munnar/test-munnar.png"
)

# Existing AI checkpoint
CHECKPOINT = Path(
    "/kaggle/working/UAV_SelfPositioning_23M_ZCN.pth"
)

print("=" * 60)
print("GENERIC DRONE GEOLOCATION PIPELINE")
print("=" * 60)

print()
print("Project root:")
print(PROJECT_ROOT)

print()
print("Directories:")

for directory in DIRECTORIES:
    print("✅", directory)

print()
print("Existing Munnar image:")

if MUNNAR_IMAGE.exists():
    print("✅", MUNNAR_IMAGE)
else:
    print("❌", MUNNAR_IMAGE)

print()
print("AI checkpoint:")

if CHECKPOINT.exists():
    print("✅", CHECKPOINT)
else:
    print("❌", CHECKPOINT)

print()
print("=" * 60)
print("PIPELINE STRUCTURE READY")
print("=" * 60)

In [ ]:
# ============================================================
# PREPARING GENERIC PIPELINE INPUTS
# ============================================================

from pathlib import Path
import shutil

PROJECT_ROOT = Path("/kaggle/working/drone_geolocation_pipeline")

SOURCE_IMAGE = Path(
    "/kaggle/input/datasets/vishalswamy2004/munnar/test-munnar.png"
)

SOURCE_CHECKPOINT = Path(
    "/kaggle/working/UAV_SelfPositioning_23M_ZCN.pth"
)

TARGET_IMAGE = PROJECT_ROOT / "input" / "drone_image.png"
TARGET_CHECKPOINT = PROJECT_ROOT / "models" / "UAV_SelfPositioning_23M_ZCN.pth"

print("=" * 60)
print("PREPARING GENERIC PIPELINE INPUTS")
print("=" * 60)

# Copy drone image
if SOURCE_IMAGE.exists():

    shutil.copy2(
        SOURCE_IMAGE,
        TARGET_IMAGE
    )

    print()
    print("✅ Drone image copied")
    print("Source :", SOURCE_IMAGE)
    print("Target :", TARGET_IMAGE)

else:

    raise FileNotFoundError(
        f"Drone image not found: {SOURCE_IMAGE}"
    )


# Copy AI checkpoint
if SOURCE_CHECKPOINT.exists():

    shutil.copy2(
        SOURCE_CHECKPOINT,
        TARGET_CHECKPOINT
    )

    print()
    print("✅ AI checkpoint copied")
    print("Source :", SOURCE_CHECKPOINT)
    print("Target :", TARGET_CHECKPOINT)

else:

    raise FileNotFoundError(
        f"AI checkpoint not found: {SOURCE_CHECKPOINT}"
    )


print()
print("=" * 60)
print("VERIFYING COPIED FILES")
print("=" * 60)

print()

if TARGET_IMAGE.exists():

    print("✅", TARGET_IMAGE)
    print("Size:", TARGET_IMAGE.stat().st_size, "bytes")

else:

    print("❌ Image copy failed")


print()

if TARGET_CHECKPOINT.exists():

    print("✅", TARGET_CHECKPOINT)
    print("Size:", TARGET_CHECKPOINT.stat().st_size, "bytes")

else:

    print("❌ Checkpoint copy failed")


print()
print("=" * 60)
print("PIPELINE INPUT PREPARATION COMPLETED")
print("=" * 60)

## 3. Download the pretrained UAV checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

model_dir = Path("/kaggle/working/drone_geolocation_pipeline/models")
model_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = hf_hub_download(
    repo_id="Bancie/UAV-Self-Positioning-23M-ZCN",
    filename="UAV_SelfPositioning_23M_ZCN.pth",
    local_dir=str(model_dir)
)

print("Checkpoint:", checkpoint_path)
print("Size (MB):", Path(checkpoint_path).stat().st_size / (1024**2))


## 4. Create the Munnar satellite reference

In [ ]:
# ============================================================
# RECREATE CORRECT MUNNAR SENTINEL-2 SATELLITE REFERENCE
# ============================================================

import os
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from pystac_client import Client

print("=" * 60)
print("RECREATING CORRECT MUNNAR SATELLITE REFERENCE")
print("=" * 60)

# ------------------------------------------------------------
# MUNNAR TARGET
# ------------------------------------------------------------

CENTER_LAT = 10.0889
CENTER_LON = 77.0595

HALF_SIZE = 0.02

WEST  = CENTER_LON - HALF_SIZE
SOUTH = CENTER_LAT - HALF_SIZE
EAST  = CENTER_LON + HALF_SIZE
NORTH = CENTER_LAT + HALF_SIZE

print()
print("Target center:")
print("Latitude :", CENTER_LAT)
print("Longitude:", CENTER_LON)

print()
print("Requested WGS84 bounds:")
print("West :", WEST)
print("South:", SOUTH)
print("East :", EAST)
print("North:", NORTH)

# ------------------------------------------------------------
# SEARCH SENTINEL-2
# ------------------------------------------------------------

catalog = Client.open(
    "https://earth-search.aws.element84.com/v1"
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[WEST, SOUTH, EAST, NORTH],
    datetime="2026-01-01/2026-12-31",
    query={
        "eo:cloud_cover": {
            "lt": 20
        }
    }
)

items = list(search.items())

print()
print("Candidate scenes:", len(items))

if len(items) == 0:
    raise RuntimeError(
        "No Sentinel-2 scenes found for the requested Munnar area."
    )

# ------------------------------------------------------------
# SELECT LOW-CLOUD SCENE
# ------------------------------------------------------------

items.sort(
    key=lambda item: item.properties.get(
        "eo:cloud_cover", 999
    )
)

item = items[0]

print()
print("=" * 60)
print("SELECTED SENTINEL-2 SCENE")
print("=" * 60)

print("ID      :", item.id)
print("Date    :", item.datetime)
print(
    "Cloud % :",
    item.properties.get("eo:cloud_cover")
)

# ------------------------------------------------------------
# CHECK RGB ASSETS
# ------------------------------------------------------------

print()
print("=" * 60)
print("CHECKING RGB ASSETS")
print("=" * 60)

required = {
    "red": "B04",
    "green": "B03",
    "blue": "B02"
}

for name, asset_key in required.items():

    if name not in item.assets:
        raise RuntimeError(
            f"{name.upper()} asset not found."
        )

    print(
        f"✅ {name.upper():5s} -> "
        f"{item.assets[name].href}"
    )

# ------------------------------------------------------------
# OPEN RED BAND TO GET SATELLITE CRS
# ------------------------------------------------------------

red_href = item.assets["red"].href
green_href = item.assets["green"].href
blue_href = item.assets["blue"].href

with rasterio.open(red_href) as src:

    satellite_crs = src.crs

    print()
    print("=" * 60)
    print("SATELLITE RASTER")
    print("=" * 60)

    print("CRS  :", satellite_crs)
    print(
        "Size :",
        src.width,
        "x",
        src.height
    )

    print("Resolution:", src.res)

    # --------------------------------------------------------
    # TRANSFORM MUNNAR BOUNDS INTO SATELLITE CRS
    # --------------------------------------------------------

    left, bottom, right, top = transform_bounds(
        "EPSG:4326",
        satellite_crs,
        WEST,
        SOUTH,
        EAST,
        NORTH
    )

    print()
    print("Munnar bounds in satellite CRS:")
    print("Left  :", left)
    print("Bottom:", bottom)
    print("Right :", right)
    print("Top   :", top)

    window = from_bounds(
        left,
        bottom,
        right,
        top,
        src.transform
    )

    window = window.round_offsets().round_lengths()

    print()
    print("Satellite pixel window:")
    print(window)

# ------------------------------------------------------------
# READ RGB WINDOW
# ------------------------------------------------------------

def read_band_window(href, window):

    with rasterio.open(href) as src:

        data = src.read(
            1,
            window=window
        )

        profile = src.profile

    return data, profile


red, profile = read_band_window(
    red_href,
    window
)

green, _ = read_band_window(
    green_href,
    window
)

blue, _ = read_band_window(
    blue_href,
    window
)

print()
print("Extracted sizes:")
print("RED  :", red.shape)
print("GREEN:", green.shape)
print("BLUE :", blue.shape)

# ------------------------------------------------------------
# STACK RGB
# ------------------------------------------------------------

rgb = np.stack(
    [red, green, blue],
    axis=0
)

print()
print("RGB shape :", rgb.shape)
print("RGB dtype :", rgb.dtype)

# ------------------------------------------------------------
# SAVE GEOTIFF
# ------------------------------------------------------------

output_path = (
    "/kaggle/working/"
    "munnar_real_satellite_correct.tif"
)

profile.update(
    driver="GTiff",
    height=rgb.shape[1],
    width=rgb.shape[2],
    count=3,
    dtype=rgb.dtype,
    crs=satellite_crs
)

with rasterio.open(
    output_path,
    "w",
    **profile
) as dst:

    dst.write(rgb)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

with rasterio.open(output_path) as check:

    print()
    print("=" * 60)
    print("FINAL SATELLITE REFERENCE")
    print("=" * 60)

    print("File :", output_path)
    print("CRS  :", check.crs)
    print(
        "Size :",
        check.width,
        "x",
        check.height
    )

    print("Bounds:", check.bounds)

    wgs84_bounds = transform_bounds(
        check.crs,
        "EPSG:4326",
        *check.bounds
    )

    print()
    print("WGS84 geographic coverage:")
    print("West :", wgs84_bounds[0])
    print("South:", wgs84_bounds[1])
    print("East :", wgs84_bounds[2])
    print("North:", wgs84_bounds[3])

print()
print("=" * 60)
print("✅ CORRECT MUNNAR SATELLITE REFERENCE RECREATED")
print("=" * 60)

## 5. Create georeferenced satellite tiles

In [ ]:
import os
import shutil
from pathlib import Path

print("=" * 60)
print("PREPARING SATELLITE INPUT FOR GENERIC PIPELINE")
print("=" * 60)

SOURCE = Path("/kaggle/working/munnar_real_satellite_correct.tif")
TARGET_DIR = Path("/kaggle/working/drone_geolocation_pipeline/satellite")
TARGET = TARGET_DIR / "munnar_satellite.tif"

print("\nChecking source satellite...")

if not SOURCE.exists():
    raise FileNotFoundError(
        f"Verified satellite file not found:\n{SOURCE}"
    )

print(f"✅ Source found: {SOURCE}")
print(f"Size: {SOURCE.stat().st_size:,} bytes")

TARGET_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(SOURCE, TARGET)

print("\n============================================================")
print("SATELLITE COPIED")
print("============================================================")
print(f"Source : {SOURCE}")
print(f"Target : {TARGET}")

if TARGET.exists():
    print(f"Size   : {TARGET.stat().st_size:,} bytes")
    print("\n✅ Satellite input ready")
else:
    raise RuntimeError("Satellite copy failed")

print("\n============================================================")
print("GENERIC PIPELINE SATELLITE DIRECTORY")
print("============================================================")

for p in sorted(TARGET_DIR.iterdir()):
    print(f"✅ {p}")

print("\n============================================================")
print("NEXT STEP READY")
print("============================================================")
print(
    "The generic pipeline can now extract 384-D satellite features "
    "from the verified Munnar satellite image."
)
print("=" * 60)

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

ROOT = Path("/kaggle/working/drone_geolocation_pipeline")
SATELLITE = ROOT / "satellite" / "munnar_satellite.tif"
TILE_DIR = ROOT / "tiles"
METADATA = TILE_DIR / "tile_metadata.csv"

TILE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("CREATING GENERIC GEOREFERENCED SATELLITE TILE GRID")
print("=" * 60)

TILE_SIZE = 128
STRIDE = 64

with rasterio.open(SATELLITE) as src:

    print("\nSatellite information:")
    print("Width       :", src.width)
    print("Height      :", src.height)
    print("CRS         :", src.crs)
    print("Resolution  :", src.res)
    print("Bands       :", src.count)
    print("Dtype       :", src.dtypes[0])
    print("Bounds      :", src.bounds)

    records = []
    tile_id = 0

    for row in range(0, src.height - TILE_SIZE + 1, STRIDE):

        for col in range(0, src.width - TILE_SIZE + 1, STRIDE):

            window = rasterio.windows.Window(
                col,
                row,
                TILE_SIZE,
                TILE_SIZE
            )

            data = src.read(
                [1, 2, 3],
                window=window
            )

            if data.shape[1] != TILE_SIZE or data.shape[2] != TILE_SIZE:
                continue

            transform = rasterio.windows.transform(
                window,
                src.transform
            )

            tile_name = f"tile_{tile_id:04d}.tif"
            tile_path = TILE_DIR / tile_name

            profile = src.profile.copy()
            profile.update(
                height=TILE_SIZE,
                width=TILE_SIZE,
                transform=transform,
                count=3
            )

            with rasterio.open(tile_path, "w", **profile) as dst:
                dst.write(data)

            center_x = col + TILE_SIZE / 2
            center_y = row + TILE_SIZE / 2

            center_lon, center_lat = rasterio.transform.xy(
                src.transform,
                center_y,
                center_x,
                offset="center"
            )

            records.append({
                "tile_id": tile_id,
                "filename": tile_name,
                "row": row,
                "col": col,
                "center_latitude": center_lat,
                "center_longitude": center_lon
            })

            tile_id += 1

df = pd.DataFrame(records)
df.to_csv(METADATA, index=False)

print("\n============================================================")
print("TILE GRID COMPLETED")
print("============================================================")
print("Tile size :", TILE_SIZE)
print("Stride    :", STRIDE)
print("Total tiles:", len(df))
print("Tile directory:", TILE_DIR)
print("Metadata:", METADATA)

print("\nFirst 10 tiles:")
print(df.head(10).to_string(index=False))

print("\n============================================================")
print("✅ GENERIC SATELLITE TILE GRID READY")
print("============================================================")

In [ ]:
import pandas as pd
import rasterio
from pathlib import Path

ROOT = Path("/kaggle/working/drone_geolocation_pipeline")
SATELLITE = ROOT / "satellite" / "munnar_satellite.tif"
TILE_DIR = ROOT / "tiles"
METADATA = TILE_DIR / "tile_metadata.csv"

print("=" * 60)
print("CORRECTING SATELLITE TILE GPS METADATA")
print("=" * 60)

TILE_SIZE = 128

records = []

with rasterio.open(SATELLITE) as src:

    print("\nSource CRS:", src.crs)
    print("Source bounds:", src.bounds)

    for tile_path in sorted(TILE_DIR.glob("tile_*.tif")):

        tile_id = int(tile_path.stem.split("_")[1])

        with rasterio.open(tile_path) as tile:

            transform = tile.transform

            # Pixel center of the 128x128 tile
            center_col = tile.width / 2.0
            center_row = tile.height / 2.0

            # Convert tile pixel center -> source CRS coordinates
            x, y = rasterio.transform.xy(
                transform,
                center_row,
                center_col,
                offset="center"
            )

            # Convert UTM EPSG:32643 -> WGS84 EPSG:4326
            lon, lat = rasterio.warp.transform(
                src.crs,
                "EPSG:4326",
                [x],
                [y]
            )

            records.append({
                "tile_id": tile_id,
                "filename": tile_path.name,
                "center_latitude": lat[0],
                "center_longitude": lon[0]
            })

df = pd.DataFrame(records).sort_values("tile_id")

df.to_csv(METADATA, index=False)

print("\n============================================================")
print("CORRECTED TILE GPS CENTERS")
print("============================================================")

print(df.head(10).to_string(index=False))

print("\n============================================================")
print("SANITY CHECK")
print("============================================================")

print(
    f"Latitude range  : "
    f"{df.center_latitude.min():.6f} -> "
    f"{df.center_latitude.max():.6f}"
)

print(
    f"Longitude range : "
    f"{df.center_longitude.min():.6f} -> "
    f"{df.center_longitude.max():.6f}"
)

print("\n============================================================")
print("EXPECTED REGION")
print("============================================================")
print("Latitude  : approximately 10.x")
print("Longitude : approximately 77.x")

print("\n============================================================")
print("METADATA SAVED")
print("============================================================")
print(METADATA)

print("\n============================================================")
print("✅ TILE GPS METADATA CORRECTED")
print("============================================================")

## 6. Extract the 384-D drone feature

In [ ]:
# ============================================================
# EXACT UAV 384-D FEATURE EXTRACTION — COMPATIBILITY FIX
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

print("=" * 60)
print("EXACT UAV 384-D FEATURE EXTRACTION")
print("DIRECT DEIT-SMALL COMPATIBLE ARCHITECTURE")
print("=" * 60)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

IMAGE_PATH = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/input/drone_image.png"
)

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/models/"
    "UAV_SelfPositioning_23M_ZCN.pth"
)

FEATURE_PATH = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/features/"
    "drone_feature_384.npy"
)

assert os.path.exists(IMAGE_PATH)
assert os.path.exists(CHECKPOINT_PATH)

# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING UAV CHECKPOINT")
print("=" * 60)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))
print("Checkpoint keys:", len(checkpoint))

# ------------------------------------------------------------
# EXTRACT EXACT BACKBONE
# ------------------------------------------------------------

backbone_state = {}

prefix = "backbone.backbone."

for key, value in checkpoint.items():

    if key.startswith(prefix):

        new_key = key[len(prefix):]

        # Classification head is not required
        if new_key.startswith("head."):
            continue

        backbone_state[new_key] = value

print("Backbone parameters:", len(backbone_state))

# ------------------------------------------------------------
# VERIFY ARCHITECTURE
# ------------------------------------------------------------

embed_dim = backbone_state["cls_token"].shape[-1]
num_tokens = backbone_state["pos_embed"].shape[1]

patch_weight = (
    backbone_state["patch_embed.proj.weight"]
)

in_chans = patch_weight.shape[1]
patch_size = patch_weight.shape[2]

block_ids = set()

for key in backbone_state:

    if key.startswith("blocks."):

        parts = key.split(".")

        if len(parts) > 1:

            try:
                block_ids.add(
                    int(parts[1])
                )
            except:
                pass

depth = len(block_ids)

mlp_dim = (
    backbone_state[
        "blocks.0.mlp.fc1.weight"
    ].shape[0]
)

mlp_ratio = mlp_dim / embed_dim

# This checkpoint architecture is known from
# the parameter shapes to be DeiT-Small.
num_heads = 6

print()
print("=" * 60)
print("CHECKPOINT ARCHITECTURE")
print("=" * 60)

print("Embedding dimension :", embed_dim)
print("Tokens              :", num_tokens)
print("Patch size          :", patch_size)
print("Input channels      :", in_chans)
print("Transformer depth   :", depth)
print("Attention heads     :", num_heads)
print("MLP ratio           :", mlp_ratio)

assert embed_dim == 384
assert num_tokens == 197
assert patch_size == 16
assert in_chans == 3
assert depth == 12
assert num_heads == 6
assert mlp_ratio == 4.0

print()
print("✅ DeiT-Small architecture confirmed")

# ------------------------------------------------------------
# DIRECT IMPORT OF DEIT IMPLEMENTATION
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING COMPATIBLE DEIT IMPLEMENTATION")
print("=" * 60)

import importlib.util
import urllib.request
import sys

DEIT_FILE = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/models/"
    "deit_models.py"
)

DEIT_URL = (
    "https://raw.githubusercontent.com/"
    "facebookresearch/deit/main/models.py"
)

if not os.path.exists(DEIT_FILE):

    print("Downloading compatible DeiT source...")

    urllib.request.urlretrieve(
        DEIT_URL,
        DEIT_FILE
    )

    print("Downloaded:", DEIT_FILE)

# ------------------------------------------------------------
# LOAD DEIT MODULE
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "deit_models",
    DEIT_FILE
)

deit_models = importlib.util.module_from_spec(
    spec
)

sys.modules["deit_models"] = deit_models

spec.loader.exec_module(
    deit_models
)

print("✅ DeiT source loaded")

# ------------------------------------------------------------
# CREATE EXACT DEIT-SMALL
# ------------------------------------------------------------

print()
print("=" * 60)
print("CREATING EXACT DEIT-SMALL")
print("=" * 60)

vit = deit_models.VisionTransformer(
    img_size=224,
    patch_size=16,
    in_chans=3,
    num_classes=1000,
    embed_dim=384,
    depth=12,
    num_heads=6,
    mlp_ratio=4.0,
    qkv_bias=True,
    norm_layer=nn.LayerNorm
)

print("Base model created")

# ------------------------------------------------------------
# REMOVE CLASSIFICATION HEAD
# ------------------------------------------------------------

vit.head = nn.Identity()

# ------------------------------------------------------------
# LOAD UAV BACKBONE
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING UAV BACKBONE WEIGHTS")
print("=" * 60)

load_result = vit.load_state_dict(
    backbone_state,
    strict=False
)

print(
    "Missing keys   :",
    len(load_result.missing_keys)
)

print(
    "Unexpected keys:",
    len(load_result.unexpected_keys)
)

print()
print("Missing keys:")

for key in load_result.missing_keys:
    print(" ", key)

print()
print("Unexpected keys:")

for key in load_result.unexpected_keys:
    print(" ", key)

# ------------------------------------------------------------
# EXPECTED RESULT
# ------------------------------------------------------------

# Only the original classification head should be absent.
# The backbone itself must load completely.

unexpected_backbone = [
    k for k in load_result.unexpected_keys
    if not k.startswith("head.")
]

missing_backbone = [
    k for k in load_result.missing_keys
    if not k.startswith("head.")
]

assert len(unexpected_backbone) == 0
assert len(missing_backbone) == 0

print()
print("✅ UAV BACKBONE WEIGHTS LOADED CORRECTLY")

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

vit = vit.to(device)
vit.eval()

# ------------------------------------------------------------
# IMAGE TRANSFORM
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

# ------------------------------------------------------------
# LOAD IMAGE
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING DRONE IMAGE")
print("=" * 60)

image = Image.open(
    IMAGE_PATH
).convert("RGB")

print("Original size:", image.size)
print("Image mode   :", image.mode)

tensor = transform(
    image
).unsqueeze(0)

tensor = tensor.to(device)

print(
    "Model input:",
    tuple(tensor.shape)
)

# ------------------------------------------------------------
# FEATURE EXTRACTION
# ------------------------------------------------------------

print()
print("=" * 60)
print("EXTRACTING EXACT 384-D FEATURE")
print("=" * 60)

with torch.no_grad():

    # DeiT forward_features returns
    # transformer tokens.

    tokens = vit.forward_features(
        tensor
    )

    print(
        "Forward feature shape:",
        tuple(tokens.shape)
    )

    # CLS token
    if tokens.ndim == 3:

        feature = tokens[:, 0, :]

    elif tokens.ndim == 2:

        feature = tokens

    else:

        raise RuntimeError(
            "Unexpected feature shape: "
            + str(tokens.shape)
        )

    # L2 normalize
    feature = F.normalize(
        feature,
        p=2,
        dim=1
    )

feature = (
    feature
    .squeeze(0)
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# VERIFY FEATURE
# ------------------------------------------------------------

print()
print("=" * 60)
print("FEATURE RESULT")
print("=" * 60)

print(
    "Feature shape:",
    feature.shape
)

print(
    "Feature norm :",
    np.linalg.norm(feature)
)

assert feature.shape == (384,)

assert abs(
    np.linalg.norm(feature) - 1.0
) < 1e-5

print()
print("First 10 feature values:")

print(
    feature[:10]
)

# ------------------------------------------------------------
# SAVE FEATURE
# ------------------------------------------------------------

os.makedirs(
    os.path.dirname(FEATURE_PATH),
    exist_ok=True
)

np.save(
    FEATURE_PATH,
    feature
)

print()
print("=" * 60)
print("FEATURE SAVED")
print("=" * 60)

print("Path:", FEATURE_PATH)

print(
    "Size:",
    os.path.getsize(
        FEATURE_PATH
    ),
    "bytes"
)

# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

loaded_feature = np.load(
    FEATURE_PATH
)

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print(
    "Loaded shape:",
    loaded_feature.shape
)

print(
    "Loaded norm :",
    np.linalg.norm(
        loaded_feature
    )
)

assert loaded_feature.shape == (384,)

assert abs(
    np.linalg.norm(
        loaded_feature
    ) - 1.0
) < 1e-5

print()
print("=" * 60)
print("✅ EXACT UAV 384-D FEATURE EXTRACTION COMPLETED")
print("=" * 60)

## 7. Extract 384-D satellite tile features

In [ ]:
# ============================================================
# GENERIC DRONE GEOLOCATION — SATELLITE 384-D FEATURE EXTRACTION
# ============================================================

import os
import sys
import csv
import numpy as np
import torch
import rasterio
from PIL import Image
from torchvision import transforms

print("=" * 60)
print("SATELLITE 384-D FEATURE EXTRACTION")
print("=" * 60)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/kaggle/working/drone_geolocation_pipeline"

SATELLITE_DIR = os.path.join(PROJECT_ROOT, "satellite")
TILES_DIR = os.path.join(PROJECT_ROOT, "tiles")
FEATURE_DIR = os.path.join(PROJECT_ROOT, "features")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(TILES_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

CHECKPOINT = os.path.join(
    MODEL_DIR,
    "UAV_SelfPositioning_23M_ZCN.pth"
)

DEIT_SOURCE = os.path.join(
    MODEL_DIR,
    "deit_models.py"
)

SATELLITE_FEATURE_FILE = os.path.join(
    FEATURE_DIR,
    "satellite_features_384.npy"
)

SATELLITE_METADATA_FILE = os.path.join(
    FEATURE_DIR,
    "satellite_feature_metadata.csv"
)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print()
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# CHECK FILES
# ------------------------------------------------------------

print()
print("=" * 60)
print("CHECKING FILES")
print("=" * 60)

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(CHECKPOINT)

if not os.path.exists(DEIT_SOURCE):
    raise FileNotFoundError(DEIT_SOURCE)

print("Checkpoint :", CHECKPOINT)
print("DeiT source:", DEIT_SOURCE)

# ------------------------------------------------------------
# LOAD COMPATIBLE DEIT SOURCE
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING COMPATIBLE DEIT")
print("=" * 60)

import importlib.util

spec = importlib.util.spec_from_file_location(
    "deit_models",
    DEIT_SOURCE
)

deit_models = importlib.util.module_from_spec(spec)

sys.modules["deit_models"] = deit_models

spec.loader.exec_module(deit_models)

print("✅ DeiT implementation loaded")

# ------------------------------------------------------------
# LOAD UAV CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 60)
print("LOADING UAV CHECKPOINT")
print("=" * 60)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu"
)

if not isinstance(checkpoint, dict):
    raise RuntimeError(
        "Unexpected checkpoint format"
    )

print("Checkpoint keys:", len(checkpoint))

# ------------------------------------------------------------
# EXTRACT BACKBONE
# ------------------------------------------------------------

backbone_state = {}

for key, value in checkpoint.items():

    if key.startswith("backbone.backbone."):
        new_key = key.replace(
            "backbone.backbone.",
            ""
        )

        backbone_state[new_key] = value

print(
    "Backbone parameters:",
    len(backbone_state)
)

# ------------------------------------------------------------
# CREATE EXACT DEIT-SMALL
# ------------------------------------------------------------

print()
print("=" * 60)
print("CREATING EXACT DEIT-SMALL 384-D")
print("=" * 60)

vit = deit_models.deit_small_patch16_224(
    pretrained=False
)

# The UAV checkpoint does not need the classification head
# for feature extraction.

if hasattr(vit, "head"):
    vit.head = torch.nn.Identity()

# ------------------------------------------------------------
# LOAD WEIGHTS
# ------------------------------------------------------------

load_result = vit.load_state_dict(
    backbone_state,
    strict=False
)

print(
    "Missing keys   :",
    len(load_result.missing_keys)
)

print(
    "Unexpected keys:",
    len(load_result.unexpected_keys)
)

print()
print("Missing keys:")
print(load_result.missing_keys)

print()
print("Unexpected keys:")
print(load_result.unexpected_keys)

# ------------------------------------------------------------
# MOVE MODEL
# ------------------------------------------------------------

vit = vit.to(device)
vit.eval()

print()
print("✅ Exact UAV backbone ready")

# ------------------------------------------------------------
# IMAGE TRANSFORM
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

# ------------------------------------------------------------
# FEATURE EXTRACTION
# ------------------------------------------------------------

@torch.no_grad()
def extract_feature(image):

    if image.mode != "RGB":
        image = image.convert("RGB")

    tensor = transform(
        image
    ).unsqueeze(0).to(device)

    output = vit.forward_features(
        tensor
    )

    # Expected:
    # (1, 197, 384)

    if output.ndim == 3:
        feature = output[:, 0, :]

    elif output.ndim == 2:
        feature = output

    else:
        raise RuntimeError(
            f"Unexpected feature shape: {output.shape}"
        )

    feature = torch.nn.functional.normalize(
        feature,
        p=2,
        dim=1
    )

    return feature.squeeze(0).cpu().numpy().astype(
        np.float32
    )

# ------------------------------------------------------------
# FIND SATELLITE FILES
# ------------------------------------------------------------

print()
print("=" * 60)
print("SEARCHING SATELLITE DATA")
print("=" * 60)

satellite_files = []

for root, dirs, files in os.walk(SATELLITE_DIR):

    for filename in files:

        lower = filename.lower()

        if lower.endswith(
            (".tif", ".tiff", ".png", ".jpg", ".jpeg")
        ):
            satellite_files.append(
                os.path.join(root, filename)
            )

satellite_files.sort()

print(
    "Satellite images found:",
    len(satellite_files)
)

if len(satellite_files) == 0:

    raise RuntimeError(
        "No satellite images found.\n"
        f"Place GeoTIFF/JPG/PNG satellite images inside:\n"
        f"{SATELLITE_DIR}"
    )

# ------------------------------------------------------------
# EXTRACT FEATURES
# ------------------------------------------------------------

print()
print("=" * 60)
print("EXTRACTING SATELLITE FEATURES")
print("=" * 60)

features = []
metadata = []

for index, filepath in enumerate(
    satellite_files,
    start=1
):

    try:

        # ----------------------------------------------------
        # READ IMAGE
        # ----------------------------------------------------

        if filepath.lower().endswith(
            (".tif", ".tiff")
        ):

            with rasterio.open(filepath) as src:

                data = src.read()

                # --------------------------------------------
                # Convert raster → RGB
                # --------------------------------------------

                if data.shape[0] >= 3:

                    rgb = np.stack(
                        [
                            data[0],
                            data[1],
                            data[2]
                        ],
                        axis=-1
                    )

                elif data.shape[0] == 1:

                    band = data[0]

                    rgb = np.stack(
                        [
                            band,
                            band,
                            band
                        ],
                        axis=-1
                    )

                else:

                    raise RuntimeError(
                        "Unsupported band count"
                    )

                rgb = rgb.astype(
                    np.float32
                )

                # --------------------------------------------
                # Normalize arbitrary raster values
                # --------------------------------------------

                rgb_min = np.nanmin(rgb)
                rgb_max = np.nanmax(rgb)

                if rgb_max > rgb_min:

                    rgb = (
                        (rgb - rgb_min)
                        /
                        (rgb_max - rgb_min)
                        * 255.0
                    )

                else:

                    rgb[:] = 0

                rgb = np.clip(
                    rgb,
                    0,
                    255
                ).astype(
                    np.uint8
                )

                image = Image.fromarray(
                    rgb,
                    mode="RGB"
                )

                # --------------------------------------------
                # Geospatial metadata
                # --------------------------------------------

                bounds = src.bounds
                crs = src.crs

                center_x = (
                    bounds.left +
                    bounds.right
                ) / 2.0

                center_y = (
                    bounds.bottom +
                    bounds.top
                ) / 2.0

                try:

                    from rasterio.warp import transform

                    if crs is not None:

                        lon, lat = transform(
                            crs,
                            "EPSG:4326",
                            [center_x],
                            [center_y]
                        )

                        center_lon = float(
                            lon[0]
                        )

                        center_lat = float(
                            lat[0]
                        )

                    else:

                        center_lat = np.nan
                        center_lon = np.nan

                except Exception:

                    center_lat = np.nan
                    center_lon = np.nan

        else:

            image = Image.open(
                filepath
            ).convert("RGB")

            center_lat = np.nan
            center_lon = np.nan

        # ----------------------------------------------------
        # AI FEATURE
        # ----------------------------------------------------

        feature = extract_feature(
            image
        )

        features.append(
            feature
        )

        metadata.append(
            [
                index - 1,
                os.path.basename(filepath),
                filepath,
                center_lat,
                center_lon
            ]
        )

        print(
            f"{index:03d}/{len(satellite_files):03d} "
            f"{os.path.basename(filepath)} "
            f"feature={feature.shape} "
            f"norm={np.linalg.norm(feature):.6f}"
        )

    except Exception as e:

        print()
        print(
            f"❌ FAILED: {filepath}"
        )

        print(
            "Reason:",
            repr(e)
        )

        raise

# ------------------------------------------------------------
# FEATURE MATRIX
# ------------------------------------------------------------

feature_matrix = np.stack(
    features,
    axis=0
).astype(
    np.float32
)

print()
print("=" * 60)
print("SATELLITE FEATURE MATRIX")
print("=" * 60)

print(
    "Shape:",
    feature_matrix.shape
)

print(
    "Feature dimension:",
    feature_matrix.shape[1]
)

# ------------------------------------------------------------
# SAVE FEATURES
# ------------------------------------------------------------

np.save(
    SATELLITE_FEATURE_FILE,
    feature_matrix
)

# ------------------------------------------------------------
# SAVE METADATA
# ------------------------------------------------------------

with open(
    SATELLITE_METADATA_FILE,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow(
        [
            "tile_id",
            "filename",
            "filepath",
            "center_latitude",
            "center_longitude"
        ]
    )

    writer.writerows(
        metadata
    )

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print()
print("=" * 60)
print("VERIFYING SAVED FEATURES")
print("=" * 60)

loaded = np.load(
    SATELLITE_FEATURE_FILE
)

print(
    "Loaded shape:",
    loaded.shape
)

print(
    "First feature norm:",
    np.linalg.norm(
        loaded[0]
    )
)

print()
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

print(
    SATELLITE_FEATURE_FILE
)

print(
    SATELLITE_METADATA_FILE
)

print()
print("=" * 60)
print("✅ SATELLITE 384-D FEATURE EXTRACTION COMPLETED")
print("=" * 60)

## 8. Satellite matching and local refinement

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from torchvision import transforms
import rasterio

ROOT = Path("/kaggle/working/drone_geolocation_pipeline")

CHECKPOINT = ROOT / "models" / "UAV_SelfPositioning_23M_ZCN.pth"
DEIT_SOURCE = ROOT / "models" / "deit_models.py"
TILE_DIR = ROOT / "tiles"
METADATA = TILE_DIR / "tile_metadata.csv"
DRONE_FEATURE = ROOT / "features" / "drone_feature_384.npy"
OUTPUT = ROOT / "features" / "satellite_tile_features_384.npy"
RESULT = ROOT / "results" / "satellite_tile_similarity.csv"

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — SATELLITE AI MATCHING")
print("=" * 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n============================================================")
print("CHECKING FILES")
print("============================================================")

required = [
    CHECKPOINT,
    DEIT_SOURCE,
    METADATA,
    DRONE_FEATURE
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")
    print("✅", p)

tile_files = sorted(TILE_DIR.glob("tile_*.tif"))

print("\nSatellite tiles:", len(tile_files))

if len(tile_files) == 0:
    raise RuntimeError("No satellite tiles found.")

# ------------------------------------------------------------
# LOAD EXACT DEIT IMPLEMENTATION
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING EXACT DEIT IMPLEMENTATION")
print("============================================================")

sys.path.insert(0, str(DEIT_SOURCE.parent))

from deit_models import deit_small_patch16_224

print("✅ DeiT implementation loaded")

# ------------------------------------------------------------
# LOAD UAV CHECKPOINT
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING UAV CHECKPOINT")
print("============================================================")

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", len(checkpoint))

backbone_state = {}

for key, value in checkpoint.items():

    if key.startswith("backbone.backbone."):
        new_key = key.replace(
            "backbone.backbone.",
            ""
        )
        backbone_state[new_key] = value

print("Backbone parameters:", len(backbone_state))

# ------------------------------------------------------------
# CREATE EXACT DEIT-SMALL
# ------------------------------------------------------------

print("\n============================================================")
print("CREATING EXACT DEIT-SMALL 384-D")
print("============================================================")

vit = deit_small_patch16_224(
    pretrained=False,
    num_classes=1000
)

load_result = vit.load_state_dict(
    backbone_state,
    strict=False
)

print("Missing keys   :", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))

print("Missing:", load_result.missing_keys)
print("Unexpected:", load_result.unexpected_keys)

# Remove classification head because we need the 384-D feature
vit.head = torch.nn.Identity()

vit = vit.to(device)
vit.eval()

print("\n✅ Exact UAV backbone ready")

# ------------------------------------------------------------
# IMAGE TRANSFORM
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# FEATURE EXTRACTION
# ------------------------------------------------------------

@torch.no_grad()
def extract_feature(image):

    if image.mode != "RGB":
        image = image.convert("RGB")

    tensor = transform(image)
    tensor = tensor.unsqueeze(0).to(device)

    output = vit.forward_features(tensor)

    # DeiT output:
    # [batch, tokens, 384]
    if output.ndim == 3:
        feature = output[:, 0, :]
    else:
        feature = output

    feature = feature.float()

    feature = torch.nn.functional.normalize(
        feature,
        p=2,
        dim=1
    )

    return feature.squeeze(0).cpu().numpy()

# ------------------------------------------------------------
# LOAD DRONE FEATURE
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING DRONE FEATURE")
print("============================================================")

drone_feature = np.load(DRONE_FEATURE).astype(np.float32)

print("Drone feature shape:", drone_feature.shape)
print("Drone feature norm :", np.linalg.norm(drone_feature))

if drone_feature.shape != (384,):
    raise RuntimeError(
        f"Expected drone feature shape (384,), "
        f"got {drone_feature.shape}"
    )

drone_feature = drone_feature / np.linalg.norm(drone_feature)

# ------------------------------------------------------------
# EXTRACT SATELLITE FEATURES
# ------------------------------------------------------------

print("\n============================================================")
print("EXTRACTING SATELLITE 384-D FEATURES")
print("============================================================")

features = []
valid_files = []

for i, tile_path in enumerate(tile_files, start=1):

    try:

        with rasterio.open(tile_path) as src:

            data = src.read([1, 2, 3])

        # uint16 -> displayable RGB
        data = data.astype(np.float32)

        for b in range(3):

            band = data[b]

            lo = np.percentile(band, 2)
            hi = np.percentile(band, 98)

            if hi > lo:
                band = (band - lo) / (hi - lo)
            else:
                band = np.zeros_like(band)

            data[b] = np.clip(
                band * 255.0,
                0,
                255
            )

        rgb = np.transpose(
            data,
            (1, 2, 0)
        ).astype(np.uint8)

        image = Image.fromarray(
            rgb,
            mode="RGB"
        )

        feature = extract_feature(image)

        features.append(feature)
        valid_files.append(tile_path.name)

        if i <= 10 or i % 10 == 0 or i == len(tile_files):

            print(
                f"Tile {i:03d}/{len(tile_files)} "
                f"{tile_path.name} "
                f"feature={feature.shape}"
            )

    except Exception as e:

        print(
            f"❌ Failed tile {tile_path.name}: {e}"
        )

# ------------------------------------------------------------
# FEATURE MATRIX
# ------------------------------------------------------------

satellite_features = np.stack(
    features
).astype(np.float32)

print("\n============================================================")
print("SATELLITE FEATURE MATRIX")
print("============================================================")

print(
    "Shape:",
    satellite_features.shape
)

print(
    "Expected:",
    (len(valid_files), 384)
)

# ------------------------------------------------------------
# SAVE FEATURES
# ------------------------------------------------------------

np.save(
    OUTPUT,
    satellite_features
)

print("\nSaved:")
print(OUTPUT)

# ------------------------------------------------------------
# COSINE SIMILARITY
# ------------------------------------------------------------

print("\n============================================================")
print("CALCULATING AI COSINE SIMILARITY")
print("============================================================")

similarities = satellite_features @ drone_feature

metadata = pd.read_csv(METADATA)

metadata = metadata[
    metadata["filename"].isin(valid_files)
].copy()

metadata["similarity"] = metadata[
    "filename"
].map(
    dict(zip(valid_files, similarities))
)

metadata = metadata.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

metadata.to_csv(
    RESULT,
    index=False
)

# ------------------------------------------------------------
# TOP RESULTS
# ------------------------------------------------------------

print("\n============================================================")
print("TOP 10 AI SATELLITE MATCHES")
print("============================================================")

print(
    metadata[
        [
            "tile_id",
            "filename",
            "center_latitude",
            "center_longitude",
            "similarity"
        ]
    ].head(10).to_string(index=False)
)

# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

best = metadata.iloc[0]

if len(metadata) > 1:
    margin = (
        float(best["similarity"])
        - float(metadata.iloc[1]["similarity"])
    )
else:
    margin = float("nan")

print("\n============================================================")
print("BEST AI GEOLOCATION")
print("============================================================")

print(
    f"Tile ID          : {int(best['tile_id'])}"
)

print(
    f"Filename         : {best['filename']}"
)

print(
    f"Latitude         : {best['center_latitude']:.9f}"
)

print(
    f"Longitude        : {best['center_longitude']:.9f}"
)

print(
    f"Cosine similarity: {best['similarity']:.6f}"
)

print(
    f"Top-1 margin     : {margin:.6f}"
)

print("\nGoogle Maps:")
print(
    "https://www.google.com/maps/search/?api=1&query="
    f"{best['center_latitude']},"
    f"{best['center_longitude']}"
)

print("\n============================================================")
print("RESULT SAVED")
print("============================================================")

print(RESULT)

print("\n============================================================")
print("✅ GENERIC DRONE GEOLOCATION AI MATCHING COMPLETED")
print("============================================================")

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import rasterio

from PIL import Image
from pathlib import Path
from torchvision import transforms
from rasterio.windows import Window
from rasterio.transform import xy


# ============================================================
# GENERIC DRONE GEOLOCATION — CORRECT LOCAL REFINEMENT
# ============================================================

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — CORRECT LOCAL REFINEMENT")
print("=" * 60)


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/drone_geolocation_pipeline")

CHECKPOINT = ROOT / "models" / "UAV_SelfPositioning_23M_ZCN.pth"
DEIT_SOURCE = ROOT / "models" / "deit_models.py"

SATELLITE = ROOT / "satellite" / "munnar_satellite.tif"

TILE_DIR = ROOT / "tiles"
METADATA = TILE_DIR / "tile_metadata.csv"

DRONE_FEATURE = (
    ROOT / "features" / "drone_feature_384.npy"
)

COARSE_RESULT = (
    ROOT / "results" /
    "satellite_tile_similarity.csv"
)

OUTPUT = (
    ROOT / "results" /
    "local_refinement_results.csv"
)


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# CHECK FILES
# ------------------------------------------------------------

print("\n============================================================")
print("CHECKING FILES")
print("============================================================")

for path in [
    CHECKPOINT,
    DEIT_SOURCE,
    SATELLITE,
    METADATA,
    DRONE_FEATURE,
    COARSE_RESULT
]:

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

    print("✅", path)


# ------------------------------------------------------------
# LOAD COARSE RESULTS
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING COARSE AI RESULTS")
print("============================================================")

coarse = pd.read_csv(
    COARSE_RESULT
)

coarse = coarse.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

TOP_K = min(
    5,
    len(coarse)
)

top_candidates = coarse.head(TOP_K)

print("\nTop candidates:")

print(
    top_candidates[
        [
            "tile_id",
            "filename",
            "center_latitude",
            "center_longitude",
            "similarity"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# LOAD DRONE FEATURE
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING DRONE 384-D FEATURE")
print("============================================================")

drone_feature = np.load(
    DRONE_FEATURE
).astype(np.float32)

if drone_feature.shape != (384,):
    raise RuntimeError(
        f"Expected (384,), got "
        f"{drone_feature.shape}"
    )

drone_feature /= np.linalg.norm(
    drone_feature
)

print(
    "Drone feature shape:",
    drone_feature.shape
)

print(
    "Drone feature norm :",
    np.linalg.norm(drone_feature)
)


# ------------------------------------------------------------
# LOAD EXACT DEIT IMPLEMENTATION
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING EXACT DEIT IMPLEMENTATION")
print("============================================================")

sys.path.insert(
    0,
    str(DEIT_SOURCE.parent)
)

from deit_models import (
    deit_small_patch16_224
)

print("✅ DeiT implementation loaded")


# ------------------------------------------------------------
# LOAD UAV CHECKPOINT
# ------------------------------------------------------------

print("\n============================================================")
print("LOADING UAV CHECKPOINT")
print("============================================================")

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

backbone_state = {}

for key, value in checkpoint.items():

    if key.startswith(
        "backbone.backbone."
    ):

        new_key = key.replace(
            "backbone.backbone.",
            ""
        )

        backbone_state[new_key] = value


print(
    "Checkpoint keys:",
    len(checkpoint)
)

print(
    "Backbone parameters:",
    len(backbone_state)
)


# ------------------------------------------------------------
# CREATE EXACT DEIT-SMALL
# ------------------------------------------------------------

print("\n============================================================")
print("CREATING EXACT DEIT-SMALL 384-D")
print("============================================================")

vit = deit_small_patch16_224(
    pretrained=False,
    num_classes=1000
)

load_result = vit.load_state_dict(
    backbone_state,
    strict=False
)

print(
    "Missing keys   :",
    len(load_result.missing_keys)
)

print(
    "Unexpected keys:",
    len(load_result.unexpected_keys)
)

if load_result.missing_keys:
    print(
        "Missing:",
        load_result.missing_keys
    )

if load_result.unexpected_keys:
    print(
        "Unexpected:",
        load_result.unexpected_keys
    )

vit.head = torch.nn.Identity()

vit = vit.to(device)

vit.eval()

print(
    "✅ Exact UAV backbone ready"
)


# ------------------------------------------------------------
# TRANSFORM
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ------------------------------------------------------------
# FEATURE EXTRACTION
# ------------------------------------------------------------

@torch.no_grad()
def extract_feature(image):

    if image.mode != "RGB":
        image = image.convert(
            "RGB"
        )

    tensor = transform(
        image
    )

    tensor = tensor.unsqueeze(
        0
    ).to(device)

    output = vit.forward_features(
        tensor
    )

    if output.ndim == 3:

        feature = output[
            :,
            0,
            :
        ]

    else:

        feature = output

    feature = feature.float()

    feature = torch.nn.functional.normalize(
        feature,
        p=2,
        dim=1
    )

    return feature[
        0
    ].cpu().numpy()


# ------------------------------------------------------------
# SATELLITE -> RGB
# ------------------------------------------------------------

def window_to_image(
    src,
    window
):

    data = src.read(
        [1, 2, 3],
        window=window
    )

    data = data.astype(
        np.float32
    )

    for band in range(3):

        channel = data[
            band
        ]

        low = np.percentile(
            channel,
            2
        )

        high = np.percentile(
            channel,
            98
        )

        if high > low:

            channel = (
                channel - low
            ) / (
                high - low
            )

        else:

            channel = np.zeros_like(
                channel
            )

        data[
            band
        ] = np.clip(
            channel * 255.0,
            0,
            255
        )

    rgb = np.transpose(
        data,
        (1, 2, 0)
    ).astype(
        np.uint8
    )

    return Image.fromarray(
        rgb
    )


# ------------------------------------------------------------
# REFINEMENT SETTINGS
# ------------------------------------------------------------

REFINE_SIZE = 64
REFINE_STRIDE = 16

print("\n============================================================")
print("REFINEMENT SETTINGS")
print("============================================================")

print(
    "Window size :",
    REFINE_SIZE
)

print(
    "Stride      :",
    REFINE_STRIDE
)


# ------------------------------------------------------------
# OPEN SATELLITE
# ------------------------------------------------------------

with rasterio.open(
    SATELLITE
) as src:

    print("\n============================================================")
    print("SATELLITE INFORMATION")
    print("============================================================")

    print(
        "CRS        :",
        src.crs
    )

    print(
        "Size       :",
        src.width,
        "x",
        src.height
    )

    print(
        "Resolution :",
        src.res
    )

    print(
        "Bounds     :",
        src.bounds
    )

    refinement_results = []

    # --------------------------------------------------------
    # REFINE EACH TOP COARSE TILE
    # --------------------------------------------------------

    for candidate_number, candidate in enumerate(
        top_candidates.itertuples(
            index=False
        ),
        start=1
    ):

        parent_tile_id = int(
            candidate.tile_id
        )

        parent_filename = (
            candidate.filename
        )

        parent_path = (
            TILE_DIR /
            parent_filename
        )

        print(
            f"\n------------------------------------------------------------"
        )

        print(
            f"REFINING CANDIDATE "
            f"{candidate_number}/{TOP_K}"
        )

        print(
            "Parent tile:",
            parent_tile_id,
            parent_filename
        )

        if not parent_path.exists():

            print(
                "⚠️ Parent tile missing"
            )

            continue

        # ----------------------------------------------------
        # FIND PARENT TILE PIXEL LOCATION
        # ----------------------------------------------------

        with rasterio.open(
            parent_path
        ) as parent:

            parent_transform = (
                parent.transform
            )

            parent_width = (
                parent.width
            )

            parent_height = (
                parent.height
            )

            # Parent tile center in source CRS
            px = parent_width / 2.0
            py = parent_height / 2.0

            parent_x, parent_y = (
                rasterio.transform.xy(
                    parent_transform,
                    py,
                    px,
                    offset="center"
                )
            )

        # Source-pixel coordinates
        parent_col, parent_row = (
            ~src.transform *
            (
                parent_x,
                parent_y
            )
        )

        parent_col = float(
            parent_col
        )

        parent_row = float(
            parent_row
        )

        # ----------------------------------------------------
        # SEARCH REGION AROUND PARENT
        # ----------------------------------------------------

        search_radius = 64

        start_col = max(
            0,
            int(parent_col)
            - search_radius
        )

        end_col = min(
            src.width - REFINE_SIZE,
            int(parent_col)
            + search_radius
        )

        start_row = max(
            0,
            int(parent_row)
            - search_radius
        )

        end_row = min(
            src.height - REFINE_SIZE,
            int(parent_row)
            + search_radius
        )

        windows = []

        for row in range(
            start_row,
            end_row + 1,
            REFINE_STRIDE
        ):

            for col in range(
                start_col,
                end_col + 1,
                REFINE_STRIDE
            ):

                windows.append(
                    (
                        row,
                        col
                    )
                )

        print(
            "Refinement windows:",
            len(windows)
        )

        # ----------------------------------------------------
        # EXTRACT FEATURES
        # ----------------------------------------------------

        for local_index, (
            row,
            col
        ) in enumerate(
            windows,
            start=1
        ):

            window = Window(
                col,
                row,
                REFINE_SIZE,
                REFINE_SIZE
            )

            try:

                image = window_to_image(
                    src,
                    window
                )

                feature = extract_feature(
                    image
                )

                similarity = float(
                    np.dot(
                        drone_feature,
                        feature
                    )
                )

                # --------------------------------------------
                # TRUE WINDOW CENTER
                # --------------------------------------------

                center_row = (
                    row +
                    REFINE_SIZE / 2.0
                )

                center_col = (
                    col +
                    REFINE_SIZE / 2.0
                )

                x, y = (
                    rasterio.transform.xy(
                        src.transform,
                        center_row,
                        center_col,
                        offset="center"
                    )
                )

                # --------------------------------------------
                # UTM -> WGS84
                # --------------------------------------------

                lon, lat = (
                    rasterio.warp.transform(
                        src.crs,
                        "EPSG:4326",
                        [x],
                        [y]
                    )
                )

                refinement_results.append({
                    "parent_tile_id":
                        parent_tile_id,

                    "parent_filename":
                        parent_filename,

                    "parent_similarity":
                        float(
                            candidate.similarity
                        ),

                    "row":
                        row,

                    "col":
                        col,

                    "window_size":
                        REFINE_SIZE,

                    "center_latitude":
                        lat[0],

                    "center_longitude":
                        lon[0],

                    "similarity":
                        similarity
                })

                if (
                    local_index <= 3
                    or
                    local_index % 25 == 0
                    or
                    local_index == len(windows)
                ):

                    print(
                        f"Window "
                        f"{local_index:03d}/"
                        f"{len(windows)} "
                        f"similarity="
                        f"{similarity:.6f}"
                    )

            except Exception as e:

                print(
                    f"⚠️ Window failed "
                    f"row={row} col={col}: "
                    f"{e}"
                )


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

refined = pd.DataFrame(
    refinement_results
)

if len(refined) == 0:

    raise RuntimeError(
        "No refinement windows were processed."
    )


refined = refined.sort_values(
    "similarity",
    ascending=False
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

refined.to_csv(
    OUTPUT,
    index=False
)


# ------------------------------------------------------------
# TOP RESULTS
# ------------------------------------------------------------

print("\n============================================================")
print("TOP 15 REFINED AI MATCHES")
print("============================================================")

print(
    refined[
        [
            "parent_tile_id",
            "center_latitude",
            "center_longitude",
            "similarity"
        ]
    ].head(15).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# FINAL ESTIMATE
# ------------------------------------------------------------

best = refined.iloc[0]

if len(refined) >= 2:

    margin = (
        float(best.similarity)
        -
        float(refined.iloc[1].similarity)
    )

else:

    margin = float("nan")


print("\n============================================================")
print("FINAL REFINED AI GEOLOCATION")
print("============================================================")

print(
    "Parent tile      :",
    int(best.parent_tile_id)
)

print(
    "Latitude         :",
    f"{best.center_latitude:.9f}"
)

print(
    "Longitude        :",
    f"{best.center_longitude:.9f}"
)

print(
    "Similarity        :",
    f"{best.similarity:.6f}"
)

print(
    "Top-1 margin      :",
    f"{margin:.6f}"
)

print("\nGoogle Maps:")

print(
    "https://www.google.com/maps/search/?api=1&query="
    f"{best.center_latitude},"
    f"{best.center_longitude}"
)


print("\n============================================================")
print("RESULT SAVED")
print("============================================================")

print(
    OUTPUT
)

print("\n============================================================")
print("✅ GENERIC LOCAL REFINEMENT COMPLETED")
print("============================================================")

In [ ]:
import os
import pandas as pd
import numpy as np

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — REFINED RESULT DEDUPLICATION")
print("=" * 60)

RESULT_FILE = "/kaggle/working/drone_geolocation_pipeline/results/local_refinement_results.csv"

assert os.path.exists(RESULT_FILE), f"Missing: {RESULT_FILE}"

df = pd.read_csv(RESULT_FILE)

print()
print("Original refined results:", len(df))

print()
print("Columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# REMOVE DUPLICATE GEOGRAPHIC WINDOWS
# ------------------------------------------------------------

# Coordinates are generated from the same raster windows.
# Round to ~1 m level before deduplication.
df["lat_key"] = df["center_latitude"].round(5)
df["lon_key"] = df["center_longitude"].round(5)

# Keep the highest similarity for each geographic location
dedup = (
    df.sort_values("similarity", ascending=False)
      .drop_duplicates(
          subset=["lat_key", "lon_key"],
          keep="first"
      )
      .copy()
)

dedup = dedup.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# REAL TOP-1 / TOP-2
# ------------------------------------------------------------

top1 = dedup.iloc[0]

if len(dedup) >= 2:
    top2 = dedup.iloc[1]
    margin = float(top1["similarity"] - top2["similarity"])
else:
    top2 = None
    margin = float("nan")

print()
print("=" * 60)
print("DEDUPLICATED TOP 10")
print("=" * 60)

display(
    dedup[
        [
            "parent_tile_id",
            "center_latitude",
            "center_longitude",
            "similarity"
        ]
    ].head(10)
)

print()
print("=" * 60)
print("REAL BEST REFINED LOCATION")
print("=" * 60)

print(f"Latitude    : {top1['center_latitude']:.9f}")
print(f"Longitude   : {top1['center_longitude']:.9f}")
print(f"Similarity  : {top1['similarity']:.6f}")

if top2 is not None:
    print()
    print("Second-best unique location:")
    print(f"Latitude    : {top2['center_latitude']:.9f}")
    print(f"Longitude   : {top2['center_longitude']:.9f}")
    print(f"Similarity  : {top2['similarity']:.6f}")
    print()
    print(f"REAL Top-1 margin : {margin:.6f}")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/results/"
    "local_refinement_results_deduplicated.csv"
)

dedup.drop(
    columns=["lat_key", "lon_key"]
).to_csv(
    OUTPUT,
    index=False
)

print()
print("=" * 60)
print("SAVED")
print("=" * 60)
print(OUTPUT)

print()
print("Google Maps:")
print(
    "https://www.google.com/maps/search/?api=1&query="
    f"{top1['center_latitude']},{top1['center_longitude']}"
)

print()
print("============================================================")
print("DEDUPLICATION COMPLETED")
print("============================================================")

In [ ]:
import os
import pandas as pd
import numpy as np

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — SPATIAL CONSISTENCY CHECK")
print("=" * 60)

INPUT = "/kaggle/working/drone_geolocation_pipeline/results/local_refinement_results_deduplicated.csv"

assert os.path.exists(INPUT), f"Missing file: {INPUT}"

df = pd.read_csv(INPUT)

df = df.sort_values("similarity", ascending=False).reset_index(drop=True)

print()
print("Unique refined locations:", len(df))

# ------------------------------------------------------------
# TOP 10
# ------------------------------------------------------------

top10 = df.head(10).copy()

print()
print("=" * 60)
print("TOP 10 UNIQUE LOCATIONS")
print("=" * 60)

display(
    top10[
        [
            "parent_tile_id",
            "center_latitude",
            "center_longitude",
            "similarity"
        ]
    ]
)

# ------------------------------------------------------------
# HAVERSINE DISTANCE
# ------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0088

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


best_lat = float(top10.iloc[0]["center_latitude"])
best_lon = float(top10.iloc[0]["center_longitude"])

top10["distance_from_best_km"] = haversine_km(
    best_lat,
    best_lon,
    top10["center_latitude"].values,
    top10["center_longitude"].values
)

print()
print("=" * 60)
print("DISTANCE FROM BEST LOCATION")
print("=" * 60)

display(
    top10[
        [
            "center_latitude",
            "center_longitude",
            "similarity",
            "distance_from_best_km"
        ]
    ]
)

# ------------------------------------------------------------
# TOP 5 CLUSTER STATISTICS
# ------------------------------------------------------------

top5 = top10.head(5)

mean_lat = top5["center_latitude"].mean()
mean_lon = top5["center_longitude"].mean()

median_lat = top5["center_latitude"].median()
median_lon = top5["center_longitude"].median()

print()
print("=" * 60)
print("TOP-5 SPATIAL CLUSTER")
print("=" * 60)

print(f"Best latitude       : {best_lat:.9f}")
print(f"Best longitude      : {best_lon:.9f}")

print()
print(f"Mean latitude       : {mean_lat:.9f}")
print(f"Mean longitude      : {mean_lon:.9f}")

print()
print(f"Median latitude     : {median_lat:.9f}")
print(f"Median longitude    : {median_lon:.9f}")

# Maximum distance among top 5
max_cluster_distance = top5["distance_from_best_km"].max()

print()
print(f"Maximum Top-5 distance from best: {max_cluster_distance:.4f} km")

# ------------------------------------------------------------
# WEIGHTED LOCATION
# ------------------------------------------------------------

weights = top5["similarity"].values

weighted_lat = np.average(
    top5["center_latitude"].values,
    weights=weights
)

weighted_lon = np.average(
    top5["center_longitude"].values,
    weights=weights
)

print()
print("=" * 60)
print("SIMILARITY-WEIGHTED LOCATION")
print("=" * 60)

print(f"Weighted latitude  : {weighted_lat:.9f}")
print(f"Weighted longitude : {weighted_lon:.9f}")

# ------------------------------------------------------------
# SAVE VALIDATION
# ------------------------------------------------------------

OUTPUT = (
    "/kaggle/working/"
    "drone_geolocation_pipeline/results/"
    "spatial_consistency_validation.csv"
)

top10.to_csv(OUTPUT, index=False)

print()
print("=" * 60)
print("VALIDATION SAVED")
print("=" * 60)

print(OUTPUT)

print()
print("=" * 60)
print("SPATIAL CONSISTENCY CHECK COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — CLUSTER VALIDATION")
print("=" * 60)

INPUT = (
    "/kaggle/working/drone_geolocation_pipeline/"
    "results/local_refinement_results_deduplicated.csv"
)

OUTPUT = (
    "/kaggle/working/drone_geolocation_pipeline/"
    "results/spatial_cluster_validation.csv"
)

assert os.path.exists(INPUT), f"Missing file: {INPUT}"

df = pd.read_csv(INPUT)

df = df.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# HAVERSINE
# ------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0088

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


# ------------------------------------------------------------
# CLUSTER RADIUS
# ------------------------------------------------------------

CLUSTER_RADIUS_KM = 0.25

print()
print("Cluster radius:", CLUSTER_RADIUS_KM, "km")
print("Candidate locations:", len(df))


# ------------------------------------------------------------
# FIND CLUSTER AROUND EACH CANDIDATE
# ------------------------------------------------------------

cluster_results = []

for i, candidate in df.iterrows():

    lat = float(candidate["center_latitude"])
    lon = float(candidate["center_longitude"])

    distances = haversine_km(
        lat,
        lon,
        df["center_latitude"].values,
        df["center_longitude"].values
    )

    mask = distances <= CLUSTER_RADIUS_KM

    members = df.loc[mask].copy()

    if len(members) == 0:
        continue

    similarities = members["similarity"].values

    # Similarity-weighted cluster center
    weighted_lat = np.average(
        members["center_latitude"].values,
        weights=similarities
    )

    weighted_lon = np.average(
        members["center_longitude"].values,
        weights=similarities
    )

    # Mean similarity
    mean_similarity = members["similarity"].mean()

    # Maximum similarity
    max_similarity = members["similarity"].max()

    # Number of supporting locations
    support_count = len(members)

    # Combined cluster score
    cluster_score = (
        max_similarity
        + 0.02 * np.log1p(support_count)
        + 0.05 * mean_similarity
    )

    cluster_results.append({
        "candidate_index": i,
        "candidate_latitude": lat,
        "candidate_longitude": lon,
        "candidate_similarity": float(
            candidate["similarity"]
        ),
        "support_count": support_count,
        "mean_similarity": mean_similarity,
        "max_similarity": max_similarity,
        "weighted_latitude": weighted_lat,
        "weighted_longitude": weighted_lon,
        "cluster_score": cluster_score
    })


# ------------------------------------------------------------
# SORT CLUSTERS
# ------------------------------------------------------------

clusters = pd.DataFrame(cluster_results)

clusters = clusters.sort_values(
    "cluster_score",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# TOP CLUSTERS
# ------------------------------------------------------------

print()
print("=" * 60)
print("TOP 10 SPATIAL CLUSTERS")
print("=" * 60)

display(
    clusters.head(10)[
        [
            "candidate_latitude",
            "candidate_longitude",
            "candidate_similarity",
            "support_count",
            "mean_similarity",
            "max_similarity",
            "weighted_latitude",
            "weighted_longitude",
            "cluster_score"
        ]
    ]
)


# ------------------------------------------------------------
# BEST CLUSTER
# ------------------------------------------------------------

best_cluster = clusters.iloc[0]

final_lat = float(
    best_cluster["weighted_latitude"]
)

final_lon = float(
    best_cluster["weighted_longitude"]
)

final_similarity = float(
    best_cluster["candidate_similarity"]
)

support = int(
    best_cluster["support_count"]
)


# ------------------------------------------------------------
# SECOND CLUSTER
# ------------------------------------------------------------

if len(clusters) > 1:

    second_cluster = clusters.iloc[1]

    second_score = float(
        second_cluster["cluster_score"]
    )

    cluster_margin = (
        float(best_cluster["cluster_score"])
        - second_score
    )

else:

    cluster_margin = float("nan")


# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print()
print("=" * 60)
print("BEST SPATIAL CLUSTER")
print("=" * 60)

print(
    f"Cluster center latitude  : {final_lat:.9f}"
)

print(
    f"Cluster center longitude : {final_lon:.9f}"
)

print(
    f"Best candidate similarity: {final_similarity:.6f}"
)

print(
    f"Supporting locations     : {support}"
)

print(
    f"Cluster score            : "
    f"{best_cluster['cluster_score']:.6f}"
)

print(
    f"Cluster-vs-second margin : "
    f"{cluster_margin:.6f}"
)


# ------------------------------------------------------------
# GOOGLE MAPS
# ------------------------------------------------------------

print()
print("Google Maps:")
print(
    f"https://www.google.com/maps/search/?api=1"
    f"&query={final_lat},{final_lon}"
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

clusters.to_csv(
    OUTPUT,
    index=False
)

print()
print("=" * 60)
print("CLUSTER VALIDATION SAVED")
print("=" * 60)

print(OUTPUT)

print()
print("=" * 60)
print("✅ SPATIAL CLUSTER VALIDATION COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — TRUE SPATIAL PEAK ANALYSIS")
print("=" * 60)

INPUT = (
    "/kaggle/working/drone_geolocation_pipeline/"
    "results/local_refinement_results_deduplicated.csv"
)

OUTPUT = (
    "/kaggle/working/drone_geolocation_pipeline/"
    "results/true_spatial_peaks.csv"
)

assert os.path.exists(INPUT), f"Missing: {INPUT}"

df = pd.read_csv(INPUT)

df = df.sort_values(
    "similarity",
    ascending=False
).reset_index(drop=True)

print()
print("Input refined locations:", len(df))


# ------------------------------------------------------------
# HAVERSINE DISTANCE
# ------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0088

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


# ------------------------------------------------------------
# NON-MAXIMUM SUPPRESSION
# ------------------------------------------------------------

SUPPRESSION_RADIUS_KM = 0.20
SUPPORT_RADIUS_KM = 0.25

remaining = df.copy()

peaks = []

while len(remaining) > 0:

    # Highest similarity remaining point
    peak = remaining.iloc[0]

    peak_lat = float(peak["center_latitude"])
    peak_lon = float(peak["center_longitude"])
    peak_similarity = float(peak["similarity"])

    distances = haversine_km(
        peak_lat,
        peak_lon,
        remaining["center_latitude"].values,
        remaining["center_longitude"].values
    )

    # --------------------------------------------------------
    # SUPPORTING LOCATIONS
    # --------------------------------------------------------

    support_mask = distances <= SUPPORT_RADIUS_KM

    support = remaining.loc[support_mask].copy()

    support_sim = support["similarity"].values

    weighted_lat = np.average(
        support["center_latitude"].values,
        weights=support_sim
    )

    weighted_lon = np.average(
        support["center_longitude"].values,
        weights=support_sim
    )

    mean_similarity = float(
        support["similarity"].mean()
    )

    median_similarity = float(
        support["similarity"].median()
    )

    support_count = int(
        len(support)
    )

    # Distance of supporting locations from peak
    support_distances = haversine_km(
        peak_lat,
        peak_lon,
        support["center_latitude"].values,
        support["center_longitude"].values
    )

    max_support_distance = float(
        support_distances.max()
    )

    peaks.append({
        "peak_latitude": peak_lat,
        "peak_longitude": peak_lon,
        "peak_similarity": peak_similarity,
        "support_count": support_count,
        "mean_similarity": mean_similarity,
        "median_similarity": median_similarity,
        "weighted_latitude": weighted_lat,
        "weighted_longitude": weighted_lon,
        "max_support_distance_km":
            max_support_distance
    })

    # --------------------------------------------------------
    # REMOVE EVERYTHING NEAR THIS PEAK
    # --------------------------------------------------------

    suppress_mask = distances > SUPPRESSION_RADIUS_KM

    remaining = remaining.loc[
        suppress_mask
    ].copy()

    remaining = remaining.sort_values(
        "similarity",
        ascending=False
    ).reset_index(drop=True)

    if len(peaks) >= 10:
        break


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

peaks_df = pd.DataFrame(peaks)

print()
print("=" * 60)
print("INDEPENDENT SPATIAL PEAKS")
print("=" * 60)

display(
    peaks_df[
        [
            "peak_latitude",
            "peak_longitude",
            "peak_similarity",
            "support_count",
            "mean_similarity",
            "median_similarity",
            "weighted_latitude",
            "weighted_longitude",
            "max_support_distance_km"
        ]
    ]
)


# ------------------------------------------------------------
# PEAK MARGINS
# ------------------------------------------------------------

if len(peaks_df) >= 2:

    peak_margin = (
        peaks_df.iloc[0]["peak_similarity"]
        - peaks_df.iloc[1]["peak_similarity"]
    )

else:

    peak_margin = np.nan


# ------------------------------------------------------------
# BEST PEAK
# ------------------------------------------------------------

best = peaks_df.iloc[0]

best_lat = float(
    best["peak_latitude"]
)

best_lon = float(
    best["peak_longitude"]
)

best_similarity = float(
    best["peak_similarity"]
)

print()
print("=" * 60)
print("BEST INDEPENDENT SPATIAL PEAK")
print("=" * 60)

print(
    f"Latitude   : {best_lat:.9f}"
)

print(
    f"Longitude  : {best_lon:.9f}"
)

print(
    f"Similarity : {best_similarity:.6f}"
)

print(
    f"Support    : {int(best['support_count'])}"
)

print(
    f"Mean support similarity : "
    f"{best['mean_similarity']:.6f}"
)

print(
    f"Median support similarity : "
    f"{best['median_similarity']:.6f}"
)

print(
    f"Peak margin vs next independent peak : "
    f"{peak_margin:.6f}"
)

print()
print("Google Maps:")
print(
    f"https://www.google.com/maps/search/?api=1"
    f"&query={best_lat},{best_lon}"
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

peaks_df.to_csv(
    OUTPUT,
    index=False
)

print()
print("=" * 60)
print("SAVED")
print("=" * 60)

print(OUTPUT)

print()
print("=" * 60)
print("✅ TRUE SPATIAL PEAK ANALYSIS COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np
import rasterio
from PIL import Image, ImageDraw, ImageFont

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — INDEPENDENT PEAK VISUAL VALIDATION")
print("=" * 60)

PIPELINE = "/kaggle/working/drone_geolocation_pipeline"

DRONE = os.path.join(
    PIPELINE,
    "input",
    "drone_image.png"
)

SATELLITE = os.path.join(
    PIPELINE,
    "satellite",
    "munnar_satellite.tif"
)

PEAKS = os.path.join(
    PIPELINE,
    "results",
    "true_spatial_peaks.csv"
)

OUTPUT = os.path.join(
    PIPELINE,
    "results",
    "independent_peak_visual_validation.png"
)

assert os.path.exists(DRONE)
assert os.path.exists(SATELLITE)
assert os.path.exists(PEAKS)

print()
print("✅ Drone image found")
print("✅ Satellite image found")
print("✅ Spatial peak results found")


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

peaks = pd.read_csv(PEAKS)

peaks = peaks.sort_values(
    "peak_similarity",
    ascending=False
).head(5)

print()
print("=" * 60)
print("PEAKS SELECTED FOR VISUAL VALIDATION")
print("=" * 60)

print(
    peaks[
        [
            "peak_latitude",
            "peak_longitude",
            "peak_similarity",
            "support_count"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# LOAD DRONE IMAGE
# ------------------------------------------------------------

drone = Image.open(DRONE).convert("RGB")

# Resize while preserving a useful display size
drone.thumbnail((500, 350))

print()
print("Drone display size:", drone.size)


# ------------------------------------------------------------
# OPEN GEOREFERENCED SATELLITE
# ------------------------------------------------------------

src = rasterio.open(SATELLITE)

print()
print("=" * 60)
print("SATELLITE")
print("=" * 60)

print("CRS        :", src.crs)
print("Size       :", src.width, "x", src.height)
print("Resolution :", src.res)


# ------------------------------------------------------------
# CONVERT LAT/LON TO PIXELS
# ------------------------------------------------------------

def latlon_to_pixel(lat, lon):

    x, y = rasterio.transform.rowcol(
        src.transform,
        lon,
        lat
    )

    return int(x), int(y)


# ------------------------------------------------------------
# EXTRACT SATELLITE PATCH
# ------------------------------------------------------------

PATCH_SIZE = 160

patches = []

for rank, (_, row) in enumerate(
    peaks.iterrows(),
    start=1
):

    lat = float(row["peak_latitude"])
    lon = float(row["peak_longitude"])
    similarity = float(row["peak_similarity"])

    pixel_row, pixel_col = latlon_to_pixel(
        lat,
        lon
    )

    half = PATCH_SIZE // 2

    r0 = max(
        0,
        pixel_row - half
    )

    r1 = min(
        src.height,
        pixel_row + half
    )

    c0 = max(
        0,
        pixel_col - half
    )

    c1 = min(
        src.width,
        pixel_col + half
    )

    patch = src.read(
        [1, 2, 3],
        window=rasterio.windows.Window(
            c0,
            r0,
            c1 - c0,
            r1 - r0
        )
    )

    patch = np.transpose(
        patch,
        (1, 2, 0)
    )

    # Convert uint16 / arbitrary satellite range
    # into displayable 8-bit RGB
    patch = patch.astype(np.float32)

    pmin = np.percentile(
        patch,
        2
    )

    pmax = np.percentile(
        patch,
        98
    )

    if pmax > pmin:

        patch = (
            (patch - pmin)
            / (pmax - pmin)
            * 255.0
        )

    patch = np.clip(
        patch,
        0,
        255
    ).astype(np.uint8)

    patch_img = Image.fromarray(
        patch,
        mode="RGB"
    )

    patch_img = patch_img.resize(
        (320, 320)
    )

    patches.append(
        (
            rank,
            lat,
            lon,
            similarity,
            patch_img
        )
    )

    print(
        f"Peak {rank}: "
        f"pixel=({pixel_row},{pixel_col}) "
        f"similarity={similarity:.6f}"
    )


src.close()


# ------------------------------------------------------------
# CREATE VALIDATION CANVAS
# ------------------------------------------------------------

CELL_W = 350
CELL_H = 390

cols = 2
rows = 3

canvas = Image.new(
    "RGB",
    (
        cols * CELL_W,
        rows * CELL_H
    ),
    "white"
)

draw = ImageDraw.Draw(canvas)


# ------------------------------------------------------------
# DRONE PANEL
# ------------------------------------------------------------

drone_panel = Image.new(
    "RGB",
    (320, 320),
    "black"
)

drone_copy = drone.copy()

x = (320 - drone_copy.width) // 2
y = (320 - drone_copy.height) // 2

drone_panel.paste(
    drone_copy,
    (x, y)
)

canvas.paste(
    drone_panel,
    (15, 15)
)

draw.text(
    (15, 340),
    "DRONE IMAGE",
    fill="black"
)


# ------------------------------------------------------------
# PEAK PANELS
# ------------------------------------------------------------

for i, (
    rank,
    lat,
    lon,
    similarity,
    patch
) in enumerate(patches):

    index = i + 1

    grid_index = index

    row_idx = grid_index // cols
    col_idx = grid_index % cols

    x0 = col_idx * CELL_W + 15
    y0 = row_idx * CELL_H + 15

    canvas.paste(
        patch,
        (x0, y0)
    )

    label = (
        f"PEAK {rank} | "
        f"sim={similarity:.4f}\n"
        f"{lat:.6f}, {lon:.6f}"
    )

    draw.multiline_text(
        (x0, y0 + 325),
        label,
        fill="black",
        spacing=4
    )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

canvas.save(
    OUTPUT
)

print()
print("=" * 60)
print("VISUAL VALIDATION SAVED")
print("=" * 60)

print(OUTPUT)

print()
print("=" * 60)
print("✅ INDEPENDENT PEAK VISUAL VALIDATION COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np
import rasterio
from PIL import Image, ImageDraw

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — INDEPENDENT PEAK VISUAL VALIDATION")
print("=" * 60)

PIPELINE = "/kaggle/working/drone_geolocation_pipeline"

DRONE = os.path.join(
    PIPELINE,
    "input",
    "drone_image.png"
)

SATELLITE = os.path.join(
    PIPELINE,
    "satellite",
    "munnar_satellite.tif"
)

PEAKS = os.path.join(
    PIPELINE,
    "results",
    "true_spatial_peaks.csv"
)

OUTPUT = os.path.join(
    PIPELINE,
    "results",
    "independent_peak_visual_validation.png"
)

assert os.path.exists(DRONE)
assert os.path.exists(SATELLITE)
assert os.path.exists(PEAKS)

print()
print("✅ Drone image found")
print("✅ Satellite image found")
print("✅ Spatial peak results found")


# ------------------------------------------------------------
# LOAD PEAK RESULTS
# ------------------------------------------------------------

peaks = pd.read_csv(PEAKS)

peaks = peaks.sort_values(
    "peak_similarity",
    ascending=False
).head(5)

print()
print("=" * 60)
print("PEAKS SELECTED FOR VISUAL VALIDATION")
print("=" * 60)

print(
    peaks[
        [
            "peak_latitude",
            "peak_longitude",
            "peak_similarity",
            "support_count"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# LOAD DRONE IMAGE
# ------------------------------------------------------------

drone = Image.open(DRONE).convert("RGB")

drone.thumbnail((500, 320))

print()
print("Drone display size:", drone.size)


# ------------------------------------------------------------
# OPEN SATELLITE
# ------------------------------------------------------------

src = rasterio.open(SATELLITE)

print()
print("=" * 60)
print("SATELLITE")
print("=" * 60)

print("CRS        :", src.crs)
print("Size       :", src.width, "x", src.height)
print("Resolution :", src.res)


# ------------------------------------------------------------
# LAT/LON -> PIXEL
# ------------------------------------------------------------

def latlon_to_pixel(lat, lon):

    row, col = rasterio.transform.rowcol(
        src.transform,
        lon,
        lat
    )

    return int(row), int(col)


# ------------------------------------------------------------
# EXTRACT SAFE SATELLITE PATCH
# ------------------------------------------------------------

PATCH_SIZE = 160

patches = []

for rank, (_, item) in enumerate(
    peaks.iterrows(),
    start=1
):

    lat = float(item["peak_latitude"])
    lon = float(item["peak_longitude"])
    similarity = float(item["peak_similarity"])
    support = int(item["support_count"])

    pixel_row, pixel_col = latlon_to_pixel(
        lat,
        lon
    )

    # --------------------------------------------------------
    # SAFE CENTER
    # --------------------------------------------------------

    pixel_row = max(
        0,
        min(src.height - 1, pixel_row)
    )

    pixel_col = max(
        0,
        min(src.width - 1, pixel_col)
    )

    half = PATCH_SIZE // 2

    # --------------------------------------------------------
    # SAFE WINDOW BOUNDS
    # --------------------------------------------------------

    r0 = max(
        0,
        pixel_row - half
    )

    r1 = min(
        src.height,
        pixel_row + half
    )

    c0 = max(
        0,
        pixel_col - half
    )

    c1 = min(
        src.width,
        pixel_col + half
    )

    width = c1 - c0
    height = r1 - r0

    if width <= 0 or height <= 0:

        print(
            f"⚠️ Peak {rank} skipped: "
            f"invalid window"
        )

        continue

    # --------------------------------------------------------
    # READ PATCH
    # --------------------------------------------------------

    patch = src.read(
        [1, 2, 3],
        window=rasterio.windows.Window(
            c0,
            r0,
            width,
            height
        )
    )

    patch = np.transpose(
        patch,
        (1, 2, 0)
    )

    # --------------------------------------------------------
    # NORMALIZE FOR DISPLAY
    # --------------------------------------------------------

    patch = patch.astype(
        np.float32
    )

    pmin = np.percentile(
        patch,
        2
    )

    pmax = np.percentile(
        patch,
        98
    )

    if pmax > pmin:

        patch = (
            (patch - pmin)
            / (pmax - pmin)
            * 255.0
        )

    patch = np.clip(
        patch,
        0,
        255
    ).astype(
        np.uint8
    )

    patch_img = Image.fromarray(
        patch,
        mode="RGB"
    )

    patch_img = patch_img.resize(
        (320, 320)
    )

    patches.append(
        {
            "rank": rank,
            "lat": lat,
            "lon": lon,
            "similarity": similarity,
            "support": support,
            "pixel_row": pixel_row,
            "pixel_col": pixel_col,
            "image": patch_img
        }
    )

    print(
        f"Peak {rank}: "
        f"pixel=({pixel_row},{pixel_col}) "
        f"window={width}x{height} "
        f"similarity={similarity:.6f}"
    )


src.close()


# ------------------------------------------------------------
# CREATE VALIDATION CANVAS
# ------------------------------------------------------------

CELL_W = 350
CELL_H = 390

COLS = 2
ROWS = 3

canvas = Image.new(
    "RGB",
    (
        COLS * CELL_W,
        ROWS * CELL_H
    ),
    "white"
)

draw = ImageDraw.Draw(canvas)


# ------------------------------------------------------------
# DRONE PANEL
# ------------------------------------------------------------

drone_panel = Image.new(
    "RGB",
    (320, 320),
    "black"
)

dx = (
    320 - drone.width
) // 2

dy = (
    320 - drone.height
) // 2

drone_panel.paste(
    drone,
    (dx, dy)
)

canvas.paste(
    drone_panel,
    (15, 15)
)

draw.text(
    (15, 340),
    "DRONE IMAGE",
    fill="black"
)


# ------------------------------------------------------------
# PEAK PANELS
# ------------------------------------------------------------

for i, peak in enumerate(patches):

    grid_index = i + 1

    row_idx = grid_index // COLS
    col_idx = grid_index % COLS

    x0 = (
        col_idx * CELL_W
        + 15
    )

    y0 = (
        row_idx * CELL_H
        + 15
    )

    canvas.paste(
        peak["image"],
        (x0, y0)
    )

    label = (
        f"PEAK {peak['rank']} | "
        f"sim={peak['similarity']:.4f}\n"
        f"{peak['lat']:.6f}, "
        f"{peak['lon']:.6f}\n"
        f"support={peak['support']}"
    )

    draw.multiline_text(
        (
            x0,
            y0 + 325
        ),
        label,
        fill="black",
        spacing=4
    )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

canvas.save(
    OUTPUT
)

print()
print("=" * 60)
print("VISUAL VALIDATION SAVED")
print("=" * 60)

print(
    OUTPUT
)

print()
print("=" * 60)
print("✅ INDEPENDENT PEAK VISUAL VALIDATION COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np
import rasterio
from rasterio.windows import Window
from pyproj import Transformer
from PIL import Image, ImageDraw

print("=" * 60)
print("GENERIC DRONE GEOLOCATION — CORRECTED PEAK VISUAL VALIDATION")
print("=" * 60)

PIPELINE = "/kaggle/working/drone_geolocation_pipeline"

DRONE = os.path.join(
    PIPELINE,
    "input",
    "drone_image.png"
)

SATELLITE = os.path.join(
    PIPELINE,
    "satellite",
    "munnar_satellite.tif"
)

PEAKS = os.path.join(
    PIPELINE,
    "results",
    "true_spatial_peaks.csv"
)

OUTPUT = os.path.join(
    PIPELINE,
    "results",
    "independent_peak_visual_validation_corrected.png"
)

assert os.path.exists(DRONE)
assert os.path.exists(SATELLITE)
assert os.path.exists(PEAKS)

print()
print("✅ Drone image found")
print("✅ Satellite image found")
print("✅ Spatial peak results found")


# ============================================================
# LOAD PEAKS
# ============================================================

peaks = pd.read_csv(PEAKS)

peaks = peaks.sort_values(
    "peak_similarity",
    ascending=False
).head(5)

print()
print("=" * 60)
print("PEAKS SELECTED")
print("=" * 60)

print(
    peaks[
        [
            "peak_latitude",
            "peak_longitude",
            "peak_similarity",
            "support_count"
        ]
    ].to_string(index=False)
)


# ============================================================
# LOAD DRONE
# ============================================================

drone = Image.open(
    DRONE
).convert("RGB")

drone.thumbnail(
    (500, 320)
)

print()
print("Drone display size:", drone.size)


# ============================================================
# OPEN SATELLITE
# ============================================================

src = rasterio.open(
    SATELLITE
)

print()
print("=" * 60)
print("SATELLITE")
print("=" * 60)

print("CRS        :", src.crs)
print("Size       :", src.width, "x", src.height)
print("Resolution :", src.res)
print("Bounds     :", src.bounds)


# ============================================================
# IMPORTANT:
# WGS84 LAT/LON -> SATELLITE CRS
# ============================================================

transformer = Transformer.from_crs(
    "EPSG:4326",
    src.crs,
    always_xy=True
)

print()
print("=" * 60)
print("COORDINATE TRANSFORMATION")
print("=" * 60)

print("Input CRS  : EPSG:4326")
print("Output CRS :", src.crs)


# ============================================================
# EXTRACT PATCHES
# ============================================================

PATCH_SIZE = 160

patches = []

for rank, (_, item) in enumerate(
    peaks.iterrows(),
    start=1
):

    lat = float(
        item["peak_latitude"]
    )

    lon = float(
        item["peak_longitude"]
    )

    similarity = float(
        item["peak_similarity"]
    )

    support = int(
        item["support_count"]
    )

    # --------------------------------------------------------
    # WGS84 -> UTM
    # --------------------------------------------------------

    utm_x, utm_y = transformer.transform(
        lon,
        lat
    )

    # --------------------------------------------------------
    # UTM -> PIXEL
    # --------------------------------------------------------

    pixel_row, pixel_col = rasterio.transform.rowcol(
        src.transform,
        utm_x,
        utm_y
    )

    pixel_row = int(pixel_row)
    pixel_col = int(pixel_col)

    print()
    print(
        f"Peak {rank}"
    )

    print(
        f"  GPS       : "
        f"{lat:.9f}, {lon:.9f}"
    )

    print(
        f"  UTM       : "
        f"{utm_x:.3f}, {utm_y:.3f}"
    )

    print(
        f"  Pixel     : "
        f"({pixel_row}, {pixel_col})"
    )

    print(
        f"  Similarity: "
        f"{similarity:.6f}"
    )

    # --------------------------------------------------------
    # CHECK PIXEL
    # --------------------------------------------------------

    if not (
        0 <= pixel_row < src.height
        and
        0 <= pixel_col < src.width
    ):

        print(
            "  ⚠️ Outside satellite raster"
        )

        continue

    # --------------------------------------------------------
    # SAFE PATCH
    # --------------------------------------------------------

    half = PATCH_SIZE // 2

    r0 = max(
        0,
        pixel_row - half
    )

    r1 = min(
        src.height,
        pixel_row + half
    )

    c0 = max(
        0,
        pixel_col - half
    )

    c1 = min(
        src.width,
        pixel_col + half
    )

    width = c1 - c0
    height = r1 - r0

    if width <= 0 or height <= 0:

        print(
            "  ⚠️ Invalid patch"
        )

        continue

    # --------------------------------------------------------
    # READ RGB
    # --------------------------------------------------------

    patch = src.read(
        [1, 2, 3],
        window=Window(
            c0,
            r0,
            width,
            height
        )
    )

    patch = np.transpose(
        patch,
        (1, 2, 0)
    )

    # --------------------------------------------------------
    # DISPLAY NORMALIZATION
    # --------------------------------------------------------

    patch = patch.astype(
        np.float32
    )

    pmin = np.percentile(
        patch,
        2
    )

    pmax = np.percentile(
        patch,
        98
    )

    if pmax > pmin:

        patch = (
            (patch - pmin)
            /
            (pmax - pmin)
            *
            255.0
        )

    patch = np.clip(
        patch,
        0,
        255
    ).astype(
        np.uint8
    )

    patch_img = Image.fromarray(
        patch
    )

    patch_img = patch_img.resize(
        (320, 320)
    )

    patches.append(
        {
            "rank": rank,
            "lat": lat,
            "lon": lon,
            "utm_x": utm_x,
            "utm_y": utm_y,
            "pixel_row": pixel_row,
            "pixel_col": pixel_col,
            "similarity": similarity,
            "support": support,
            "image": patch_img
        }
    )


src.close()


# ============================================================
# VISUAL CANVAS
# ============================================================

CELL_W = 350
CELL_H = 390

COLS = 2
ROWS = 3

canvas = Image.new(
    "RGB",
    (
        COLS * CELL_W,
        ROWS * CELL_H
    ),
    "white"
)

draw = ImageDraw.Draw(
    canvas
)


# ============================================================
# DRONE PANEL
# ============================================================

drone_panel = Image.new(
    "RGB",
    (320, 320),
    "black"
)

dx = (
    320 - drone.width
) // 2

dy = (
    320 - drone.height
) // 2

drone_panel.paste(
    drone,
    (dx, dy)
)

canvas.paste(
    drone_panel,
    (15, 15)
)

draw.text(
    (15, 340),
    "DRONE IMAGE",
    fill="black"
)


# ============================================================
# PEAK PANELS
# ============================================================

for i, peak in enumerate(
    patches
):

    grid_index = i + 1

    row_idx = grid_index // COLS
    col_idx = grid_index % COLS

    x0 = (
        col_idx * CELL_W
        + 15
    )

    y0 = (
        row_idx * CELL_H
        + 15
    )

    canvas.paste(
        peak["image"],
        (x0, y0)
    )

    label = (
        f"PEAK {peak['rank']} | "
        f"sim={peak['similarity']:.4f}\n"
        f"{peak['lat']:.6f}, "
        f"{peak['lon']:.6f}\n"
        f"support={peak['support']}"
    )

    draw.multiline_text(
        (
            x0,
            y0 + 325
        ),
        label,
        fill="black",
        spacing=4
    )


# ============================================================
# SAVE
# ============================================================

canvas.save(
    OUTPUT
)

print()
print("=" * 60)
print("CORRECTED VISUAL VALIDATION SAVED")
print("=" * 60)

print(
    OUTPUT
)

print()
print("=" * 60)
print("✅ CORRECTED PEAK VISUAL VALIDATION COMPLETED")
print("=" * 60)

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("FINAL GENERIC DRONE GEOLOCATION PIPELINE REPORT")
print("=" * 70)

ROOT = Path(
    "/kaggle/working/drone_geolocation_pipeline"
)

RESULTS = ROOT / "results"
FEATURES = ROOT / "features"
INPUT = ROOT / "input"
MODELS = ROOT / "models"
SATELLITE = ROOT / "satellite"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

drone_image = INPUT / "drone_image.png"
checkpoint = MODELS / "UAV_SelfPositioning_23M_ZCN.pth"
satellite_image = SATELLITE / "munnar_satellite.tif"

drone_feature = (
    FEATURES / "drone_feature_384.npy"
)

satellite_features = (
    FEATURES /
    "satellite_tile_features_384.npy"
)

coarse_results = (
    RESULTS /
    "satellite_tile_similarity.csv"
)

refined_results = (
    RESULTS /
    "local_refinement_results.csv"
)

dedup_results = (
    RESULTS /
    "local_refinement_results_deduplicated.csv"
)

peaks_results = (
    RESULTS /
    "true_spatial_peaks.csv"
)

visual_validation = (
    RESULTS /
    "independent_peak_visual_validation_corrected.png"
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print()
print("=" * 70)
print("PIPELINE FILE VERIFICATION")
print("=" * 70)

files_to_check = [
    ("Drone image", drone_image),
    ("AI checkpoint", checkpoint),
    ("Satellite image", satellite_image),
    ("Drone feature", drone_feature),
    ("Satellite features", satellite_features),
    ("Coarse matching", coarse_results),
    ("Local refinement", refined_results),
    ("Deduplicated results", dedup_results),
    ("Spatial peaks", peaks_results),
    ("Visual validation", visual_validation),
]

for name, path in files_to_check:

    if path.exists():

        print(
            f"✅ {name}: {path}"
        )

    else:

        print(
            f"❌ MISSING {name}: {path}"
        )


# ------------------------------------------------------------
# LOAD FINAL PEAKS
# ------------------------------------------------------------

peaks = pd.read_csv(
    peaks_results
)

peaks = peaks.sort_values(
    "peak_similarity",
    ascending=False
).reset_index(drop=True)

best = peaks.iloc[0]

second = (
    peaks.iloc[1]
    if len(peaks) > 1
    else None
)


# ------------------------------------------------------------
# FINAL LOCATION
# ------------------------------------------------------------

final_lat = float(
    best["peak_latitude"]
)

final_lon = float(
    best["peak_longitude"]
)

final_similarity = float(
    best["peak_similarity"]
)

support_count = int(
    best["support_count"]
)

mean_support_similarity = float(
    best["mean_similarity"]
)

median_support_similarity = float(
    best["median_similarity"]
)


if second is not None:

    second_lat = float(
        second["peak_latitude"]
    )

    second_lon = float(
        second["peak_longitude"]
    )

    second_similarity = float(
        second["peak_similarity"]
    )

    top1_margin = (
        final_similarity
        -
        second_similarity
    )

else:

    second_lat = np.nan
    second_lon = np.nan
    second_similarity = np.nan
    top1_margin = np.nan


# ------------------------------------------------------------
# GOOGLE MAPS
# ------------------------------------------------------------

google_maps_url = (
    "https://www.google.com/maps/search/"
    "?api=1&query="
    f"{final_lat},{final_lon}"
)


# ------------------------------------------------------------
# FINAL RESULT TABLE
# ------------------------------------------------------------

final_result = pd.DataFrame(
    [
        {
            "final_latitude": final_lat,
            "final_longitude": final_lon,
            "similarity": final_similarity,
            "top1_margin": top1_margin,
            "support_count": support_count,
            "mean_support_similarity":
                mean_support_similarity,
            "median_support_similarity":
                median_support_similarity,
            "second_best_latitude":
                second_lat,
            "second_best_longitude":
                second_lon,
            "second_best_similarity":
                second_similarity,
            "google_maps_url":
                google_maps_url,
        }
    ]
)


# ------------------------------------------------------------
# SAVE FINAL CSV
# ------------------------------------------------------------

final_csv = (
    RESULTS /
    "final_drone_geolocation_result.csv"
)

final_result.to_csv(
    final_csv,
    index=False
)


# ------------------------------------------------------------
# SAVE TOP PEAKS
# ------------------------------------------------------------

top_peaks_csv = (
    RESULTS /
    "final_top_spatial_peaks.csv"
)

peaks.head(10).to_csv(
    top_peaks_csv,
    index=False
)


# ------------------------------------------------------------
# TEXT REPORT
# ------------------------------------------------------------

report_file = (
    RESULTS /
    "FINAL_DRONE_GEOLOCATION_REPORT.txt"
)

with open(
    report_file,
    "w"
) as f:

    f.write(
        "=" * 70 + "\n"
    )

    f.write(
        "FINAL DRONE GEOLOCATION REPORT\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        "INPUT\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        f"Drone image       : {drone_image}\n"
    )

    f.write(
        f"Satellite image   : {satellite_image}\n"
    )

    f.write(
        f"AI checkpoint     : {checkpoint}\n\n"
    )

    f.write(
        "FINAL AI ESTIMATE\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        f"Latitude          : {final_lat:.9f}\n"
    )

    f.write(
        f"Longitude         : {final_lon:.9f}\n"
    )

    f.write(
        f"Similarity        : {final_similarity:.6f}\n"
    )

    f.write(
        f"Top-1 margin      : {top1_margin:.6f}\n"
    )

    f.write(
        f"Support count     : {support_count}\n"
    )

    f.write(
        f"Mean support sim. : "
        f"{mean_support_similarity:.6f}\n"
    )

    f.write(
        f"Median support   : "
        f"{median_support_similarity:.6f}\n\n"
    )

    f.write(
        "SECOND-BEST PEAK\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        f"Latitude          : {second_lat:.9f}\n"
    )

    f.write(
        f"Longitude         : {second_lon:.9f}\n"
    )

    f.write(
        f"Similarity        : {second_similarity:.6f}\n\n"
    )

    f.write(
        "GOOGLE MAPS\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        google_maps_url + "\n\n"
    )

    f.write(
        "VALIDATION\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        "Embedded GPS metadata: NO\n"
    )

    f.write(
        "Therefore the reported coordinate is an "
        "AI-estimated location, not independently "
        "verified ground-truth GPS.\n\n"
    )

    f.write(
        "VISUAL VALIDATION\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        f"{visual_validation}\n\n"
    )

    f.write(
        "TOP 10 SPATIAL PEAKS\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        peaks.head(10).to_string(
            index=False
        )
    )

    f.write("\n\n")

    f.write(
        "=" * 70 + "\n"
    )

    f.write(
        "END OF REPORT\n"
    )

    f.write(
        "=" * 70 + "\n"
    )


# ------------------------------------------------------------
# FINAL CONSOLE OUTPUT
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL AI GEOLOCATION")
print("=" * 70)

print(
    f"Latitude   : {final_lat:.9f}"
)

print(
    f"Longitude  : {final_lon:.9f}"
)

print(
    f"Similarity : {final_similarity:.6f}"
)

print(
    f"Top-1 margin: {top1_margin:.6f}"
)

print(
    f"Support    : {support_count}"
)

print()
print("Second-best location:")

print(
    f"Latitude   : {second_lat:.9f}"
)

print(
    f"Longitude  : {second_lon:.9f}"
)

print(
    f"Similarity : {second_similarity:.6f}"
)

print()
print("Google Maps:")
print(
    google_maps_url
)

print()
print("=" * 70)
print("FINAL FILES")
print("=" * 70)

print(
    final_csv
)

print(
    top_peaks_csv
)

print(
    report_file
)

print(
    visual_validation
)

print()
print("=" * 70)
print("✅ FINAL GENERIC DRONE GEOLOCATION REPORT COMPLETED")
print("=" * 70)

## 9. Verified Munnar ground-truth validation

In [ ]:
from math import radians, sin, cos, sqrt, atan2
from pathlib import Path
import pandas as pd

print("=" * 70)
print("MUNNAR — FINAL GROUND-TRUTH VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# PRESERVED AI PREDICTION
# ------------------------------------------------------------

pred_lat = 10.085902557
pred_lon = 77.051077853

pred_similarity = 0.613770
pred_margin = 0.013995
support_count = 48

# ------------------------------------------------------------
# GROUND TRUTH FROM VERIFIED GOOGLE MAPS COORDINATE
# Screenshot coordinate:
# 10°04'38.2"N 77°03'29.6"E
# ------------------------------------------------------------

gt_lat = 10 + (4 / 60) + (38.2 / 3600)
gt_lon = 77 + (3 / 60) + (29.6 / 3600)

print()
print("GROUND TRUTH")
print("-" * 70)

print("Source coordinate : 10°04'38.2\"N 77°03'29.6\"E")
print(f"Latitude          : {gt_lat:.12f}")
print(f"Longitude         : {gt_lon:.12f}")

print()
print("AI PREDICTION")
print("-" * 70)

print(f"Latitude          : {pred_lat:.12f}")
print(f"Longitude         : {pred_lon:.12f}")
print(f"Similarity        : {pred_similarity:.6f}")
print(f"Top-1 margin      : {pred_margin:.6f}")
print(f"Support count     : {support_count}")

# ------------------------------------------------------------
# HAVERSINE DISTANCE
# ------------------------------------------------------------

R = 6371008.8  # metres

lat1 = radians(gt_lat)
lon1 = radians(gt_lon)
lat2 = radians(pred_lat)
lon2 = radians(pred_lon)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    sin(dlat / 2) ** 2
    + cos(lat1)
    * cos(lat2)
    * sin(dlon / 2) ** 2
)

c = 2 * atan2(
    sqrt(a),
    sqrt(1 - a)
)

error_m = R * c
error_km = error_m / 1000.0

# ------------------------------------------------------------
# DIFFERENCES
# ------------------------------------------------------------

latitude_difference = pred_lat - gt_lat
longitude_difference = pred_lon - gt_lon

print()
print("=" * 70)
print("LOCALIZATION ERROR")
print("=" * 70)

print(f"Latitude difference  : {latitude_difference:.9f}°")
print(f"Longitude difference : {longitude_difference:.9f}°")
print(f"Haversine error      : {error_m:.3f} m")
print(f"Haversine error      : {error_km:.3f} km")

# ------------------------------------------------------------
# SAVE RESULT
# ------------------------------------------------------------

result_dir = Path(
    "/kaggle/working/drone_geolocation_pipeline/results"
)

result_dir.mkdir(
    parents=True,
    exist_ok=True
)

result = pd.DataFrame([{
    "location": "Munnar",
    "ground_truth_source": "Google Maps coordinate from verified screenshot",
    "ground_truth_latitude": gt_lat,
    "ground_truth_longitude": gt_lon,
    "predicted_latitude": pred_lat,
    "predicted_longitude": pred_lon,
    "cosine_similarity": pred_similarity,
    "top1_margin": pred_margin,
    "support_count": support_count,
    "latitude_difference_deg": latitude_difference,
    "longitude_difference_deg": longitude_difference,
    "localization_error_m": error_m,
    "localization_error_km": error_km
}])

csv_path = result_dir / "munnar_final_ground_truth_validation.csv"

result.to_csv(
    csv_path,
    index=False
)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

report_path = result_dir / "MUNNAR_FINAL_GEOLOCATION_REPORT.txt"

with open(report_path, "w") as f:

    f.write("=" * 70 + "\n")
    f.write("FINAL MUNNAR DRONE GEOLOCATION VALIDATION REPORT\n")
    f.write("=" * 70 + "\n\n")

    f.write("GROUND TRUTH\n")
    f.write("-" * 70 + "\n")
    f.write("Source coordinate : 10°04'38.2\"N 77°03'29.6\"E\n")
    f.write(f"Latitude          : {gt_lat:.12f}\n")
    f.write(f"Longitude         : {gt_lon:.12f}\n\n")

    f.write("AI PREDICTION\n")
    f.write("-" * 70 + "\n")
    f.write(f"Latitude          : {pred_lat:.12f}\n")
    f.write(f"Longitude         : {pred_lon:.12f}\n")
    f.write(f"Cosine similarity : {pred_similarity:.6f}\n")
    f.write(f"Top-1 margin      : {pred_margin:.6f}\n")
    f.write(f"Support count     : {support_count}\n\n")

    f.write("LOCALIZATION ERROR\n")
    f.write("-" * 70 + "\n")
    f.write(f"Latitude error    : {latitude_difference:.9f} degrees\n")
    f.write(f"Longitude error   : {longitude_difference:.9f} degrees\n")
    f.write(f"Haversine error   : {error_m:.3f} m\n")
    f.write(f"Haversine error   : {error_km:.3f} km\n\n")

    f.write("=" * 70 + "\n")
    f.write("MUNNAR VALIDATION COMPLETED\n")
    f.write("=" * 70 + "\n")

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(csv_path)
print(report_path)

print()
print("=" * 70)
print("✅ MUNNAR FINAL GROUND-TRUTH VALIDATION COMPLETED")
print("=" * 70)

## 10. Final verified project summary

In [ ]:
import os
import csv
from datetime import datetime

# ============================================================
# MUNNAR — FINAL PROJECT DOCUMENTATION PACKAGE
# ============================================================

results = "/kaggle/working/drone_geolocation_pipeline/results"
os.makedirs(results, exist_ok=True)

# ------------------------------------------------------------
# VERIFIED FINAL VALUES
# ------------------------------------------------------------

ground_truth_lat = 10.077277777778
ground_truth_lon = 77.058222222222

prediction_lat = 10.085902557000
prediction_lon = 77.051077853000

similarity = 0.613770
top1_margin = 0.013995
support_count = 48

latitude_difference = prediction_lat - ground_truth_lat
longitude_difference = prediction_lon - ground_truth_lon

localization_error_m = 1237.539
localization_error_km = 1.237539

# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------

drone_image = (
    "/kaggle/working/drone_geolocation_pipeline/input/drone_image.png"
)

checkpoint = (
    "/kaggle/working/drone_geolocation_pipeline/models/"
    "UAV_SelfPositioning_23M_ZCN.pth"
)

satellite_image = (
    "/kaggle/working/drone_geolocation_pipeline/satellite/"
    "munnar_satellite.tif"
)

visual_validation = (
    "/kaggle/working/drone_geolocation_pipeline/results/"
    "MUNNAR_FINAL_GEOLOCATION_VALIDATION.png"
)

# ------------------------------------------------------------
# FINAL CSV
# ------------------------------------------------------------

csv_path = os.path.join(
    results,
    "MUNNAR_FINAL_PROJECT_SUMMARY.csv"
)

rows = [{
    "location": "Munnar",
    "ground_truth_latitude": ground_truth_lat,
    "ground_truth_longitude": ground_truth_lon,
    "prediction_latitude": prediction_lat,
    "prediction_longitude": prediction_lon,
    "latitude_difference_deg": latitude_difference,
    "longitude_difference_deg": longitude_difference,
    "cosine_similarity": similarity,
    "top1_margin": top1_margin,
    "spatial_support_count": support_count,
    "localization_error_m": localization_error_m,
    "localization_error_km": localization_error_km
}]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

# ------------------------------------------------------------
# FINAL TEXT REPORT
# ------------------------------------------------------------

report_path = os.path.join(
    results,
    "MUNNAR_FINAL_PROJECT_REPORT.txt"
)

report = f"""
======================================================================
MUNNAR DRONE GEOLOCATION — FINAL PROJECT REPORT
======================================================================

PROJECT
----------------------------------------------------------------------
Task:
Estimate the geographic location of a Munnar drone image by matching
its visual features against satellite imagery.

======================================================================
INPUT
======================================================================

Drone image:
{drone_image}

Satellite reference:
{satellite_image}

AI checkpoint:
{checkpoint}

======================================================================
GROUND TRUTH
======================================================================

Source:
10°04'38.2"N 77°03'29.6"E

Latitude:
{ground_truth_lat:.12f}

Longitude:
{ground_truth_lon:.12f}

======================================================================
AI PREDICTION
======================================================================

Latitude:
{prediction_lat:.12f}

Longitude:
{prediction_lon:.12f}

Cosine similarity:
{similarity:.6f}

Top-1 margin:
{top1_margin:.6f}

Spatial support:
{support_count} locations

======================================================================
POSITION DIFFERENCE
======================================================================

Latitude difference:
{latitude_difference:.12f} degrees

Longitude difference:
{longitude_difference:.12f} degrees

======================================================================
LOCALIZATION ACCURACY
======================================================================

Haversine localization error:
{localization_error_m:.3f} metres

Haversine localization error:
{localization_error_km:.6f} km

Reported project error:
{localization_error_km:.3f} km

======================================================================
INTERPRETATION
======================================================================

The AI system localized the Munnar drone image to a point
approximately {localization_error_km:.3f} km from the verified
ground-truth coordinate.

The prediction was obtained from the highest-scoring refined
spatial peak, supported by {support_count} nearby candidate
locations.

======================================================================
FINAL RESULT
======================================================================

Ground truth:
{ground_truth_lat:.9f}, {ground_truth_lon:.9f}

AI prediction:
{prediction_lat:.9f}, {prediction_lon:.9f}

Localization error:
{localization_error_km:.3f} km

======================================================================
VISUAL VALIDATION
======================================================================

{visual_validation}

======================================================================
CONCLUSION
======================================================================

The Munnar drone geolocation experiment was successfully completed.
The pipeline produced a geographic prediction and was subsequently
validated against an independently supplied ground-truth coordinate.

Final localization error:
{localization_error_km:.3f} km

======================================================================
END OF REPORT
======================================================================
"""

with open(report_path, "w") as f:
    f.write(report.strip())

# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

print("=" * 70)
print("MUNNAR — FINAL PROJECT PACKAGE")
print("=" * 70)

print()
print("Ground truth:")
print(f"  {ground_truth_lat:.9f}, {ground_truth_lon:.9f}")

print()
print("AI prediction:")
print(f"  {prediction_lat:.9f}, {prediction_lon:.9f}")

print()
print(f"Similarity       : {similarity:.6f}")
print(f"Top-1 margin     : {top1_margin:.6f}")
print(f"Spatial support  : {support_count}")

print()
print(f"Localization error:")
print(f"  {localization_error_m:.3f} m")
print(f"  {localization_error_km:.3f} km")

print()
print("FINAL FILES")
print("-" * 70)
print(csv_path)
print(report_path)
print(visual_validation)

print()
print("=" * 70)
print("✅ MUNNAR FINAL PROJECT PACKAGE COMPLETED")
print("=" * 70)

## 11. Final drone + AI satellite visualization

In [ ]:
# ================================================================
# MUNNAR — COMPLETE FINAL DRONE GEOLOCATION VISUALIZATION
#
# Includes:
#   1. Original drone image
#   2. Real Esri World Imagery satellite region
#   3. AI estimated location marker
#   4. Ground-truth coordinate
#   5. AI coordinate
#   6. Localization error
#   7. LIVE Google Maps links
#   8. Final PNG
#   9. Final CSV
#
# IMPORTANT:
# This cell DOES NOT rerun the AI model.
# It uses the already verified project results.
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import math
import io
import csv
import requests
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display, Markdown


# ================================================================
# PROJECT PATHS
# ================================================================

BASE = "/kaggle/working/drone_geolocation_pipeline"

RESULTS = os.path.join(
    BASE,
    "results"
)

os.makedirs(
    RESULTS,
    exist_ok=True
)

IMAGE_PATH = (
    "/kaggle/input/datasets/"
    "vishalswamy2004/munnar/"
    "test-munnar.png"
)

FINAL_IMAGE_PATH = os.path.join(
    RESULTS,
    "MUNNAR_FINAL_DRONE_VS_AI_SATELLITE.png"
)

FINAL_CSV_PATH = os.path.join(
    RESULTS,
    "MUNNAR_FINAL_GEOLOCATION_RESULTS.csv"
)


# ================================================================
# VERIFIED PROJECT RESULTS
# ================================================================

# Ground-truth coordinate
GT_LAT = 10.077277778
GT_LON = 77.058222222

# AI estimated coordinate
PRED_LAT = 10.085902557
PRED_LON = 77.051077853

# Verified matching metrics
SIMILARITY = 0.613770
TOP1_MARGIN = 0.013995
SUPPORT = 48

# Verified localization error
ERROR_M = 1237.539
ERROR_KM = 1.238

# AI feature dimension
FEATURE_DIM = 384


# ================================================================
# GOOGLE MAPS LIVE URLS
# ================================================================

# Ground-truth location
GT_MAPS_URL = (
    "https://www.google.com/maps/search/"
    "?api=1"
    f"&query={GT_LAT},{GT_LON}"
)

# AI-estimated location
AI_MAPS_URL = (
    "https://www.google.com/maps/search/"
    "?api=1"
    f"&query={PRED_LAT},{PRED_LON}"
)

# Satellite-view Google Maps URL for AI estimate
AI_SATELLITE_MAPS_URL = (
    "https://www.google.com/maps/@"
    "?api=1"
    "&map_action=map"
    f"&center={PRED_LAT},{PRED_LON}"
    "&zoom=15"
    "&basemap=satellite"
)

# Satellite-view Google Maps URL for ground truth
GT_SATELLITE_MAPS_URL = (
    "https://www.google.com/maps/@"
    "?api=1"
    "&map_action=map"
    f"&center={GT_LAT},{GT_LON}"
    "&zoom=15"
    "&basemap=satellite"
)


# ================================================================
# LAT/LON → WEB MERCATOR TILE
# ================================================================

ZOOM = 15
TILE_SIZE = 256
GRID_RADIUS = 1


def latlon_to_tile(lat, lon, zoom):

    lat_rad = math.radians(lat)

    n = 2 ** zoom

    x = (
        (lon + 180.0)
        / 360.0
        * n
    )

    y = (
        (
            1.0
            - math.asinh(
                math.tan(lat_rad)
            ) / math.pi
        )
        / 2.0
        * n
    )

    return x, y


# ================================================================
# DOWNLOAD ESRI WORLD IMAGERY
# ================================================================

def get_satellite_image(
    lat,
    lon,
    zoom=15,
    radius=1
):

    center_x, center_y = latlon_to_tile(
        lat,
        lon,
        zoom
    )

    center_x = int(center_x)
    center_y = int(center_y)

    grid_size = (
        2 * radius + 1
    )

    canvas = Image.new(
        "RGB",
        (
            grid_size * TILE_SIZE,
            grid_size * TILE_SIZE
        )
    )

    session = requests.Session()

    for dy in range(
        -radius,
        radius + 1
    ):

        for dx in range(
            -radius,
            radius + 1
        ):

            tile_x = center_x + dx
            tile_y = center_y + dy

            url = (
                "https://server.arcgisonline.com/"
                "ArcGIS/rest/services/"
                "World_Imagery/MapServer/tile/"
                f"{zoom}/{tile_y}/{tile_x}"
            )

            print(
                f"Downloading tile "
                f"x={tile_x}, y={tile_y}"
            )

            response = session.get(
                url,
                timeout=30
            )

            response.raise_for_status()

            tile = Image.open(
                io.BytesIO(
                    response.content
                )
            ).convert("RGB")

            px = (
                dx + radius
            ) * TILE_SIZE

            py = (
                dy + radius
            ) * TILE_SIZE

            canvas.paste(
                tile,
                (px, py)
            )

    # Exact AI coordinate location
    fx, fy = latlon_to_tile(
        lat,
        lon,
        zoom
    )

    pixel_x = (
        fx
        - (center_x - radius)
    ) * TILE_SIZE

    pixel_y = (
        fy
        - (center_y - radius)
    ) * TILE_SIZE

    return (
        canvas,
        pixel_x,
        pixel_y
    )


# ================================================================
# START
# ================================================================

print("=" * 80)
print("MUNNAR — COMPLETE FINAL GEOLOCATION RESULT")
print("=" * 80)


# ================================================================
# CHECK ORIGINAL IMAGE
# ================================================================

if not os.path.exists(
    IMAGE_PATH
):

    raise FileNotFoundError(
        f"\nMunnar image not found:\n"
        f"{IMAGE_PATH}"
    )


drone_img = Image.open(
    IMAGE_PATH
).convert("RGB")

drone_np = np.array(
    drone_img
)


# ================================================================
# PRINT BASIC INFORMATION
# ================================================================

print("\nORIGINAL DRONE IMAGE")
print("-" * 80)

print(
    f"Path       : {IMAGE_PATH}"
)

print(
    f"Resolution : "
    f"{drone_img.size[0]} × "
    f"{drone_img.size[1]}"
)

print(
    f"Mode       : {drone_img.mode}"
)


# ================================================================
# PRINT GROUND TRUTH
# ================================================================

print("\nGROUND TRUTH")
print("-" * 80)

print(
    f"Latitude  : {GT_LAT:.9f}"
)

print(
    f"Longitude : {GT_LON:.9f}"
)


# ================================================================
# PRINT AI ESTIMATE
# ================================================================

print("\nAI ESTIMATED LOCATION")
print("-" * 80)

print(
    f"Latitude  : {PRED_LAT:.9f}"
)

print(
    f"Longitude : {PRED_LON:.9f}"
)


# ================================================================
# PRINT MATCHING INFORMATION
# ================================================================

print("\nAI MATCHING RESULTS")
print("-" * 80)

print(
    f"Feature dimension : {FEATURE_DIM}"
)

print(
    f"Cosine similarity : {SIMILARITY:.6f}"
)

print(
    f"Top-1 margin      : {TOP1_MARGIN:.6f}"
)

print(
    f"Spatial support   : {SUPPORT}"
)


# ================================================================
# PRINT ERROR
# ================================================================

print("\nLOCALIZATION ERROR")
print("-" * 80)

print(
    f"Error : {ERROR_M:.3f} m"
)

print(
    f"Error : {ERROR_KM:.3f} km"
)


# ================================================================
# DOWNLOAD ACTUAL SATELLITE REGION
# ================================================================

print("\n" + "=" * 80)
print("DOWNLOADING ACTUAL ESRI WORLD IMAGERY")
print("=" * 80)

sat_img, marker_x, marker_y = (
    get_satellite_image(
        PRED_LAT,
        PRED_LON,
        zoom=ZOOM,
        radius=GRID_RADIUS
    )
)

print("\n✅ Satellite region downloaded")

print(
    f"Satellite size : "
    f"{sat_img.size[0]} × "
    f"{sat_img.size[1]}"
)

print(
    f"AI marker pixel : "
    f"({marker_x:.1f}, {marker_y:.1f})"
)


# ================================================================
# CREATE FINAL FIGURE
# ================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 8)
)


# ------------------------------------------------
# LEFT: ORIGINAL DRONE IMAGE
# ------------------------------------------------

axes[0].imshow(
    drone_np
)

axes[0].set_title(
    "Original Munnar Drone Image",
    fontsize=15
)

axes[0].axis("off")


# ------------------------------------------------
# RIGHT: ACTUAL SATELLITE IMAGE
# ------------------------------------------------

axes[1].imshow(
    sat_img
)

axes[1].scatter(
    marker_x,
    marker_y,
    s=350,
    marker="*",
    label="AI Estimated Location"
)

axes[1].set_title(
    "Satellite Region at AI Estimated Location",
    fontsize=15
)

axes[1].legend(
    loc="upper right"
)

axes[1].axis("off")


# ------------------------------------------------
# MAIN TITLE
# ------------------------------------------------

fig.suptitle(
    "Munnar Drone Geolocation — "
    "Original Image vs AI Estimated Satellite Region",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()


# ================================================================
# SAVE BEFORE SHOW
# ================================================================

print("\n" + "=" * 80)
print("SAVING FINAL VISUALIZATION")
print("=" * 80)

fig.savefig(
    FINAL_IMAGE_PATH,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"Saved to : {FINAL_IMAGE_PATH}"
)


# ================================================================
# VERIFY FINAL IMAGE
# ================================================================

if not os.path.exists(
    FINAL_IMAGE_PATH
):

    raise RuntimeError(
        "❌ Final visualization was not saved."
    )

saved_img = Image.open(
    FINAL_IMAGE_PATH
)

file_size = os.path.getsize(
    FINAL_IMAGE_PATH
)

print(
    f"Exists   : True"
)

print(
    f"File size: {file_size:,} bytes"
)

print(
    f"Image    : {saved_img.size}"
)

print(
    f"Mode     : {saved_img.mode}"
)

print(
    "\n✅ FINAL VISUALIZATION VERIFIED"
)


# ================================================================
# DISPLAY FIGURE
# ================================================================

plt.show()


# ================================================================
# CREATE FINAL CSV
# ================================================================

csv_rows = [

    [
        "Parameter",
        "Value"
    ],

    [
        "Original Image",
        "test-munnar.png"
    ],

    [
        "Image Resolution",
        "1920 × 1080"
    ],

    [
        "Feature Dimension",
        FEATURE_DIM
    ],

    [
        "Ground Truth Latitude",
        f"{GT_LAT:.12f}"
    ],

    [
        "Ground Truth Longitude",
        f"{GT_LON:.12f}"
    ],

    [
        "AI Estimated Latitude",
        f"{PRED_LAT:.12f}"
    ],

    [
        "AI Estimated Longitude",
        f"{PRED_LON:.12f}"
    ],

    [
        "Cosine Similarity",
        f"{SIMILARITY:.6f}"
    ],

    [
        "Top-1 Margin",
        f"{TOP1_MARGIN:.6f}"
    ],

    [
        "Spatial Support",
        SUPPORT
    ],

    [
        "Localization Error (m)",
        f"{ERROR_M:.3f}"
    ],

    [
        "Localization Error (km)",
        f"{ERROR_KM:.3f}"
    ],

    [
        "Satellite Source",
        "Esri World Imagery"
    ],

    [
        "Satellite Zoom",
        ZOOM
    ],

    [
        "Ground Truth Google Maps",
        GT_MAPS_URL
    ],

    [
        "AI Estimate Google Maps",
        AI_MAPS_URL
    ],

    [
        "AI Satellite Google Maps",
        AI_SATELLITE_MAPS_URL
    ]

]


with open(
    FINAL_CSV_PATH,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerows(
        csv_rows
    )


# ================================================================
# GOOGLE MAPS — DISPLAY CLICKABLE LINKS
# ================================================================

display(
    Markdown(
        f"""
# 🗺️ MUNNAR — LIVE GOOGLE MAPS LOCATIONS

### 📍 Ground-Truth / Original Location

**Coordinates:**  
`{GT_LAT:.9f}, {GT_LON:.9f}`

👉 **[OPEN LIVE GOOGLE MAPS — GROUND TRUTH]({GT_MAPS_URL})**

---

### 🤖 AI Estimated Location

**Coordinates:**  
`{PRED_LAT:.9f}, {PRED_LON:.9f}`

👉 **[OPEN LIVE GOOGLE MAPS — AI ESTIMATE]({AI_MAPS_URL})**

---

### 🛰️ AI Estimate — Satellite View

👉 **[OPEN LIVE GOOGLE MAPS — SATELLITE VIEW]({AI_SATELLITE_MAPS_URL})**

---

### 🛰️ Ground Truth — Satellite View

👉 **[OPEN LIVE GOOGLE MAPS — GROUND TRUTH SATELLITE VIEW]({GT_SATELLITE_MAPS_URL})**
"""
    )
)


# ================================================================
# FINAL TERMINAL SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("MUNNAR — FINAL PROJECT SUMMARY")
print("=" * 80)

print(
    f"""
Original drone image:
    {IMAGE_PATH}

Ground-truth coordinate:
    {GT_LAT:.9f}, {GT_LON:.9f}

AI estimated coordinate:
    {PRED_LAT:.9f}, {PRED_LON:.9f}

Cosine similarity:
    {SIMILARITY:.6f}

Top-1 margin:
    {TOP1_MARGIN:.6f}

Spatial support:
    {SUPPORT}

Localization error:
    {ERROR_M:.3f} m
    {ERROR_KM:.3f} km

Feature dimension:
    {FEATURE_DIM}

Satellite source:
    Esri World Imagery

Satellite zoom:
    {ZOOM}

Final visualization:
    {FINAL_IMAGE_PATH}

Final CSV:
    {FINAL_CSV_PATH}

Ground-truth Google Maps:
    {GT_MAPS_URL}

AI estimated Google Maps:
    {AI_MAPS_URL}

AI satellite Google Maps:
    {AI_SATELLITE_MAPS_URL}

Ground-truth satellite Google Maps:
    {GT_SATELLITE_MAPS_URL}
"""
)

print("=" * 80)
print("✅ COMPLETE MUNNAR GEOLOCATION VISUALIZATION FINISHED")
print("=" * 80)

## 12. Ground-truth vs AI graph

In [ ]:
# ================================================================
# MUNNAR — GROUND TRUTH VS AI ESTIMATE GRAPH
# ================================================================

import os
import math
import matplotlib.pyplot as plt

# ------------------------------------------------
# VERIFIED COORDINATES
# ------------------------------------------------

GT_LAT = 10.077277778
GT_LON = 77.058222222

PRED_LAT = 10.085902557
PRED_LON = 77.051077853

ERROR_M = 1237.539
ERROR_KM = 1.238

RESULTS = "/kaggle/working/drone_geolocation_pipeline/results"

os.makedirs(
    RESULTS,
    exist_ok=True
)

GRAPH_PATH = os.path.join(
    RESULTS,
    "MUNNAR_GROUND_TRUTH_VS_AI_GRAPH.png"
)

# ------------------------------------------------
# CREATE GRAPH
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(12, 8)
)

# Ground truth
ax.scatter(
    GT_LON,
    GT_LAT,
    s=250,
    marker="o",
    label="Ground Truth"
)

# AI estimate
ax.scatter(
    PRED_LON,
    PRED_LAT,
    s=300,
    marker="*",
    label="AI Estimated Location"
)

# Connection line
ax.plot(
    [GT_LON, PRED_LON],
    [GT_LAT, PRED_LAT],
    linewidth=2,
    linestyle="--",
    label=f"Localization Error = {ERROR_KM:.3f} km"
)

# ------------------------------------------------
# LABELS
# ------------------------------------------------

ax.annotate(
    "Ground Truth\n"
    f"({GT_LAT:.9f}, {GT_LON:.9f})",
    xy=(GT_LON, GT_LAT),
    xytext=(12, -25),
    textcoords="offset points",
    fontsize=11
)

ax.annotate(
    "AI Estimate\n"
    f"({PRED_LAT:.9f}, {PRED_LON:.9f})",
    xy=(PRED_LON, PRED_LAT),
    xytext=(12, 10),
    textcoords="offset points",
    fontsize=11
)

# ------------------------------------------------
# TITLE
# ------------------------------------------------

ax.set_title(
    "Munnar Drone Geolocation — Ground Truth vs AI Estimate",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel(
    "Longitude (degrees)",
    fontsize=12
)

ax.set_ylabel(
    "Latitude (degrees)",
    fontsize=12
)

# Grid
ax.grid(
    True,
    linestyle="--",
    alpha=0.5
)

# Legend
ax.legend(
    loc="best"
)

# ------------------------------------------------
# INFORMATION BOX
# ------------------------------------------------

info_text = (
    f"Ground Truth : {GT_LAT:.9f}, {GT_LON:.9f}\n"
    f"AI Estimate  : {PRED_LAT:.9f}, {PRED_LON:.9f}\n"
    f"Error        : {ERROR_M:.3f} m ({ERROR_KM:.3f} km)"
)

ax.text(
    0.02,
    0.02,
    info_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(
        boxstyle="round,pad=0.5",
        alpha=0.15
    )
)

plt.tight_layout()

# ------------------------------------------------
# SAVE
# ------------------------------------------------

fig.savefig(
    GRAPH_PATH,
    dpi=300,
    bbox_inches="tight"
)

# ------------------------------------------------
# VERIFY
# ------------------------------------------------

print("=" * 80)
print("MUNNAR — GROUND TRUTH VS AI GRAPH")
print("=" * 80)

print(
    "\nGround Truth:"
)

print(
    f"Latitude  : {GT_LAT:.9f}"
)

print(
    f"Longitude : {GT_LON:.9f}"
)

print(
    "\nAI Estimate:"
)

print(
    f"Latitude  : {PRED_LAT:.9f}"
)

print(
    f"Longitude : {PRED_LON:.9f}"
)

print(
    f"\nLocalization Error : "
    f"{ERROR_M:.3f} m"
)

print(
    f"Localization Error : "
    f"{ERROR_KM:.3f} km"
)

print(
    "\nGraph saved to:"
)

print(
    GRAPH_PATH
)

print(
    "\nExists :",
    os.path.exists(GRAPH_PATH)
)

if os.path.exists(GRAPH_PATH):

    print(
        "Size   :",
        f"{os.path.getsize(GRAPH_PATH):,} bytes"
    )

    print(
        "\n✅ GROUND TRUTH VS AI GRAPH SAVED"
    )

plt.show()

## Final verified values

- Ground truth: **10.077277778, 77.058222222**
- AI estimate: **10.085902557, 77.051077853**
- Cosine similarity: **0.613770**
- Top-1 margin: **0.013995**
- Spatial support: **48**
- Localization error: **1237.539 m (1.238 km)**

These values are retained from the verified final Munnar result in the original notebook.